<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.fr/cap07/cap07.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

[comment]: # (Aucun texte naturel à traduire n'a été fourni dans le contenu Markdown ci-dessus — uniquement des images et des liens, qui doivent être conservés tels quels.)

## 💻 **Partie Pratique avec des Exercices de Programmation**

La présente liste d'exercices de programmation (EP) consolide les formulations théoriques présentées tout au long du Chapitre 7 — Classification d'Images et Reconnaissance de Formes — à travers un parcours pratique appliqué. Contrairement à la manipulation directe des pixels des chapitres précédents, les EP de ce chapitre travaillent avec les **grandeurs intermédiaires** d'un *pipeline* réel de reconnaissance de formes — vecteurs de caractéristiques, distances, étiquettes prédites et réelles, codes binaires locaux et histogrammes d'orientation — permettant de valider manuellement chaque étape du raisonnement sans dépendre de bibliothèques externes d'apprentissage automatique.

L'enchaînement des exercices reproduit le flux conceptuel du chapitre : on commence par l'implémentation manuelle de la règle de décision du classificateur **k-NN** sur un petit espace de caractéristiques ; ensuite, on revisite, sous l'angle de la normalisation des caractéristiques, le classificateur implémenté dans le premier exercice de la liste ; on avance vers le calcul des métriques d'**évaluation** (matrice de confusion, précision et rappel) à partir des étiquettes prédites et réelles ; on poursuit avec le codage manuel du descripteur de texture **LBP** à partir d'un voisinage $3\times3$ ; on approfondit le calcul de l'histogramme des orientations du descripteur **HOG** pour une seule cellule ; on avance ensuite vers l'intégration de l'**extraction de descripteurs**, de la **classification k-NN** et de l'**évaluation multi-classe** dans un *pipeline* complet de reconnaissance de textures ; et on conclut avec l'application de ce même *pipeline* sur une **image réelle** (format PGM), où le descripteur LBP est calculé directement sur les pixels d'une mosaïque de textures.

### 🎯 Objectif de ce Carnet

Le carnet permet de développer, valider, organiser et tester des solutions d'**Exercices de Programmation (EPs)** dans des environnements interactifs, comme Colab, avec les mêmes cas de test que Moodle, en les y copiant uniquement au moment d'enregistrer la note officielle.

### *Téléchargement*

Téléchargez `morph.py` et `testsuite.py` en exécutant la cellule ci-dessous :

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Exécution des tests
Pour évaluer les tests, exécutez `TestSuite("EP07_01.extensão").run()` dans une nouvelle cellule, en remplaçant l’extension par celle du langage utilisé (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). Le système télécharge les cas de test depuis GitHub, exécute le programme et calcule automatiquement la note.

Pour tester directement le code Python, sans enregistrer de fichier, utilisez `run_code(codigo)` en passant le code sous forme de *chaîne de caractères* dans une variable `codigo` :

```python
codigo = """
# ... votre code ici ...
"""
TestSuite("EP07_01").run_code(codigo)
```

### 🛠️ Résumé des méthodes de `morph.py` (Chap. 7)

La bibliothèque `morph.py` propose deux versions pour la plupart des algorithmes : une version **didactique** (méthodes se terminant par `0`), implémentée pas à pas en NumPy, et une version **classique**, basée sur les bibliothèques `scikit-learn` et `scikit-image`. Les implémentations didactiques sont utilisées dans les **Exercices de Programmation (EPs)**, car elles ne dépendent pas de bibliothèques externes et s’exécutent dans la limite de mémoire de l’environnement **VPL** de Moodle. Les versions classiques, quant à elles, sont plus efficaces et recommandées pour des expériences dans des environnements tels que Colab et Jupyter Notebook, mais elles ne peuvent généralement **pas être utilisées dans les EPs** de Moodle, car la bibliothèque `scikit-learn` dépasse la mémoire disponible dans le VPL.

1. **Lecture des données (`readClasses`, `readDataset`, `readTrain`, `readTest`)**  
   Elles normalisent l’entrée des ensembles d’entraînement et de test, en retournant les matrices de caractéristiques ($X$) et les vecteurs d’étiquettes ($y$).

2. **Classification (`knn0` / `knn`)**  
   Elles implémentent l’algorithme des **k-plus proches voisins (k-NN)** pour la classification binaire et multiclasse, en utilisant la distance euclidienne ou de Manhattan.

3. **Normalisation (`zscore0` / `zscore`)**  
   Elles appliquent la normalisation *z-score* aux attributs, réduisant les différences d’échelle avant la classification.

4. **Évaluation (`confusion0` / `confusion`)**  
   Elles calculent la matrice de confusion et des métriques telles que l’exactitude, la précision et le rappel, tant pour les problèmes binaires que multiclasses.

5. **Descripteur de texture (`lbp0` / `lbp`)**  
   Elles calculent le ***Local Binary Pattern* (LBP)**, permettant d’obtenir la carte LBP, le code d’un pixel ou l’histogramme d’une région de l’image.

6. **Descripteur de forme (`hog0` / `hog`)**  
   Elles calculent le ***Histogram of Oriented Gradients* (HOG)**, produisant des histogrammes des orientations des gradients pour représenter les informations de forme et de contour.

### EP07_01 🟢 Classifieur k-NN Pas à Pas

Le `KNeighborsClassifier` de `scikit-learn`, utilisé tout au long du chapitre, dissimule derrière un simple appel (`.fit` / `.predict`) une règle de décision très simple : pour chaque nouvelle observation, calculer la distance à tous les exemples d'entraînement, sélectionner les $k$ plus proches et voter pour la classe majoritaire parmi eux.

Avant de vous fier à la bibliothèque, vous êtes chargé d'implémenter cette règle de zéro, pour un espace de caractéristiques bidimensionnel, exactement comme le simulateur interactif de frontière de décision du chapitre le fait en interne à chaque clic de l'utilisateur.

#### 📋 Directives d'implémentation

1. **Quantité et paramètre :** Lire l'entier $N$ (nombre d'exemples d'entraînement) et l'entier impair $k$ (nombre de voisins).
2. **Exemples d'entraînement :** Pour chacun des $N$ exemples, lire trois valeurs : les coordonnées $x$ et $y$ (réelles) et l'étiquette $r$ (entier, $0$ ou $1$).
3. **Requêtes :** Lire l'entier $Q$ (nombre de points de requête) puis les coordonnées $x_q$, $y_q$ (réelles) de chaque requête.
4. **Distance :** Pour chaque requête, calculer la distance euclidienne à **tous** les exemples d'entraînement :
$$
d(x_q, x_i) = \sqrt{(x_q - x_i)^2 + (y_q - y_i)^2}.
$$
5. **Sélection des voisins :** Trier les exemples par distance croissante et sélectionner les $k$ premiers. En cas d'**égalité de distance** à la frontière du k-ième voisin, départager par l'exemple lu **en premier** dans l'entrée (ordre de lecture stable).
6. **Vote majoritaire :** Compter les votes de chaque classe parmi les $k$ voisins sélectionnés. S'il y a **égalité dans le vote** (seulement possible lorsque $k$ est pair, ce qui ne devrait pas se produire selon la directive du point 1, mais traitez-le défensivement), attribuez la classe du voisin le plus proche parmi les classes à égalité.
7. **Sortie :** Pour chaque requête, dans l'ordre d'entrée, imprimer la classe prédite. À la fin, imprimer le total de requêtes classées comme classe `1`.

#### 📌 Contraintes computationnelles

* **Métrique fixe :** utilisez exclusivement la distance euclidienne (pas la *distance au carré*) pour le tri, bien que le résultat de la comparaison soit le même.
* **k toujours impair :** l'entrée garantit $k$ impair et $k \le N$ ; néanmoins, implémentez le départage du point 6 par robustesse.
* **Stabilité :** lors du tri par distance, préservez l'ordre relatif des exemples ayant la même distance (tri stable).

#### 🧠 Fondement théorique

| Élément | Rôle dans le k-NN |
|---|---|
| Espace de caractéristiques | Ensemble de tous les vecteurs $(x, y)$ possibles |
| Distance euclidienne | Mesure de similarité entre observations |
| $k$ petit | Frontière irrégulière, variance élevée |
| $k$ grand | Frontière lisse, biais élevé |
| Vote majoritaire | Règle de décision $\hat y = \operatorname{moda}\{y_i : x_i \in N_k(x)\}$ |

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : entiers $N$ et $k$, séparés par un espace.
* Les $N$ lignes suivantes : trois valeurs par ligne — $x$, $y$ (réelles) et $r$ (entier $\in \{0,1\}$), séparées par un espace.
* Ligne suivante : entier $Q$.
* Les $Q$ lignes suivantes : deux valeurs par ligne — $x_q$, $y_q$ (réelles), séparées par un espace.

**Sortie :**

* $Q$ lignes, chacune avec la classe prédite (`0` ou `1`) pour la requête respective, dans l'ordre d'entrée.
* Dernière ligne : `Total classe 1 : X`.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 4 3<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>1<br>1 1 | 0<br>Total classe 1 : 0 | Requête proche du groupe de classe 0. |
| 4 1<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>2<br>0.9 0.1<br>5.5 5.1 | 0<br>1<br>Total classe 1 : 1 | Avec $k=1$, chaque requête hérite de la classe du voisin le plus proche. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0701" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0701 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0701 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0701 button:hover { background: #e8dfcf; }
  #sim-ep0701 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0701_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0701_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP07_01 : Classificateur k-NN pas à pas</span>
  <span class="sim-ep0701_pill">Vote majoritaire</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0701_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Nombre de voisins (k) : <span id="sim-ep0701_vl" style="font-family:monospace; color:#26241d;">3</span>
      </label>
    </div>
    
    <input id="sim-ep0701_sl" type="range" min="1" max="7" step="2" value="3">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajustez k et voyez quels exemples d'entraînement (triés par distance) participent au vote pour la requête fixe (&starf; à x = 3, y = 3).
    </div>
  </div>

  <!-- Cards de Amostras -->
  <div id="sim-ep0701_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0701_debug" class="sim-ep0701_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep01(root){
    if (!root || root.dataset.sim07Ep01Init) return;
    root.dataset.sim07Ep01Init = "1";

    var query = {x: 3, y: 3};
    var pontos = [
      {nome: "A", x: 0, y: 0, r: 0},
      {nome: "B", x: 1, y: 0, r: 0},
      {nome: "C", x: 5, y: 5, r: 1},
      {nome: "D", x: 6, y: 5, r: 1},
      {nome: "E", x: 2, y: 2, r: 0},
      {nome: "F", x: 4, y: 4, r: 1},
      {nome: "G", x: 0, y: 2, r: 0},
      {nome: "H", x: 6, y: 3, r: 1}
    ];

    pontos.forEach(function(p, i){
      p.d = Math.sqrt(Math.pow(p.x - query.x, 2) + Math.pow(p.y - query.y, 2));
      p.idx = i;
    });

    pontos.sort(function(a, b){
      return (a.d - b.d) || (a.idx - b.idx);
    });

    var slEl  = root.querySelector('#sim-ep0701_sl');
    var vlEl  = root.querySelector('#sim-ep0701_vl');
    var cards = root.querySelector('#sim-ep0701_cards');
    var dbg   = root.querySelector('#sim-ep0701_debug');

    function render(){
      var k = parseInt(slEl.value, 10);
      vlEl.textContent = k;
      cards.innerHTML = '';
      var votos = [0, 0];

      pontos.forEach(function(p, i){
        var dentro = i < k;
        if (dentro) votos[p.r]++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? (p.r === 0 
                ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
                : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;') 
            : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;');

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + p.nome + ' (r = ' + p.r + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">d = ' + p.d.toFixed(2) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'VOTA' : '&ndash;') + '</div>';

        cards.appendChild(div);
      });

      var previsto = votos[1] > votos[0] ? 1 : (votos[0] > votos[1] ? 0 : pontos[0].r);

      if (previsto === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'k = ' + k + '  |  Votos Classe 0: ' + votos[0] + ', Classe 1: ' + votos[1] + '  |  Classe prevista: ' + previsto;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim07Ep01(){
    var root = document.getElementById('sim-ep0701');
    if (root) initSim07Ep01(root); else setTimeout(tryInitSim07Ep01, 200);
  }
  tryInitSim07Ep01();
})();
</script>
""")

**Figure 7.1:** Simulateur EP07_01 : Classifieur k-NN Pas à Pas


In [ ]:
%%writefile EP07_01.py
# Code Python

In [ ]:
TestSuite("EP07_01.py").run()

### EP07_02 🟡 Normalisation *Z-score* et Robustesse du k-NN à des Échelles Distinctes

Cet exercice revisite le classificateur implémenté dans l'**EP07_01**, cette fois sous l'angle discuté dans la section *L'Impact de l'Échelle et la Normalisation des Caractéristiques* du chapitre : le k-NN décide en se basant sur la distance entre les vecteurs, de sorte qu'une caractéristique mesurée sur une échelle beaucoup plus grande que les autres tend à **dominer** le calcul de la distance, même lorsqu'elle n'est pas la plus pertinente pour séparer les classes.

Un système d'inspection enregistre, pour chaque pièce, sa **superficie** (en pixels, pouvant atteindre des centaines ou des milliers) et sa **circularité** (toujours entre $0$ et $1$). Vous êtes chargé de classer de nouvelles pièces par k-NN de deux manières — avec et sans la standardisation *Z-score* présentée dans le chapitre — et de rapporter dans quels cas les deux approches **divergent**.

#### 📋 Directives d'Implémentation

1. **Quantité et paramètre :** Lire l'entier $N$ (nombre d'exemples d'entraînement) et l'entier impair $k$.
2. **Exemples d'entraînement :** Pour chacun des $N$ exemples, lire trois valeurs : la superficie $x_1$ (réelle), la circularité $x_2$ (réelle) et l'étiquette $r$ (entier, $0$ ou $1$).
3. **Requêtes :** Lire l'entier $Q$ puis les coordonnées $x_1, x_2$ de chaque requête.
4. **Classification sans normalisation :** Pour chaque requête, classifiez-la par k-NN directement sur $(x_1, x_2)$, avec la distance euclidienne et les mêmes règles de départage que l'EP07_01 (ordre de lecture pour les distances à égalité ; voisin le plus proche entre les classes à égalité lors du vote).
5. **Paramètres de normalisation :** Calculer la moyenne $\mu_j$ et l'écart-type **populationnel** $\sigma_j$ (division par $N$, non par $N-1$ — la même convention adoptée par la classe `StandardScaler`) de chaque caractéristique $j \in \{1,2\}$, **exclusivement sur l'ensemble d'entraînement**.
6. **Standardisation :** Transformez chaque caractéristique d'entraînement et de requête par
$$
z_j = \frac{x_j - \mu_j}{\sigma_j}.
$$
Si $\sigma_j = 0$ (caractéristique constante dans l'entraînement), définissez $z_j = 0$ pour tous les échantillons de cette caractéristique, évitant ainsi la division par zéro.
7. **Classification avec normalisation :** Répétez la classification k-NN du point 4, désormais sur les vecteurs standardisés $(z_1, z_2)$, avec les mêmes règles de départage.
8. **Sortie :** Pour chaque requête, dans l'ordre d'entrée, imprimer les deux classes prédites. À la fin, imprimer le nombre de requêtes où les deux classifications **divergent**.

#### 📌 Contraintes Computationnelles

* **Ajustement uniquement sur l'entraînement :** $\mu_j$ et $\sigma_j$ sont calculés uniquement à partir de l'ensemble d'entraînement et réappliqués aux requêtes — jamais recalculés à partir de celles-ci. Cette pratique évite la **fuite de données** (*data leakage*), mentionnée dans la section sur la normalisation du chapitre.
* **Écart-type populationnel :** utilisez $\sigma_j = \sqrt{\frac{1}{N}\sum_i (x_{i,j}-\mu_j)^2}$, et non la version échantillonnale (division par $N-1$).
* **Caractéristique constante :** traitez $\sigma_j = 0$ comme un cas spécial (point 6) ; aucune erreur de division par zéro ne doit survenir.
* **Règles de départage :** réutilisez exactement les conventions de l'EP07_01, tant pour la sélection des $k$ voisins que pour le vote majoritaire.

#### 🧠 Fondements Théoriques

| Élément | Rôle |
|---|---|
| Standardisation *Z-score* | Rééchelonne chaque caractéristique pour une moyenne de $0$ et un écart-type de $1$, rendant les échelles hétérogènes comparables |
| Ajustement (*fit*) uniquement sur l'entraînement | Garantit que l'évaluation sur les requêtes reflète uniquement ce que le modèle a appris lors de l'entraînement |
| Distance euclidienne sans normalisation | Dominée par la caractéristique de plus grande amplitude — ici, la superficie |
| Prédiction divergente | Met en évidence que l'échelle des caractéristiques, et pas seulement l'algorithme ou les données, peut déterminer la frontière de décision du k-NN |

Cet exercice renforce, de manière contrôlée, la raison pour laquelle le `StandardScaler` est appliqué avant le k-NN tout au long du chapitre : sans cette étape, les caractéristiques de circularité — même étant hautement discriminatives — peuvent être pratiquement ignorées par le classificateur face à une caractéristique de superficie avec une amplitude des centaines de fois plus grande.

#### 📦 Spécification d'Entrée et de Sortie (VPL)

**Entrée :**

* Ligne 1 : Entiers $N$ et $k$, séparés par un espace.
* Les $N$ lignes suivantes : trois valeurs par ligne — $x_1$, $x_2$ (réelles) et $r$ (entier $\in \{0,1\}$), séparées par un espace.
* Ligne suivante : entier $Q$.
* Les $Q$ lignes suivantes : deux valeurs par ligne — $x_1$, $x_2$ (réelles) de la requête, séparées par un espace.

**Sortie :**

* $Q$ lignes, au format `SemNorm=<0|1> ComNorm=<0|1>`, dans l'ordre d'entrée des requêtes.
* Dernière ligne : `Divergiu: <int>`.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 4 3<br>10 0.9 0<br>12 0.85 0<br>900 0.2 1<br>950 0.25 1<br>1<br>500 0.88 | SemNorm=1 ComNorm=0<br>Divergiu: 1 | Sans normalisation, la superficie (échelle des centaines) domine la distance et la requête est classée comme classe `1`. Après standardisation, la circularité — beaucoup plus proche des échantillons de classe `0` — commence à peser de manière comparable, et la prédiction change pour `0`. |
| 2 1<br>0 0.5 0<br>100 0.5 1<br>1<br>60 0.5 | SemNorm=1 ComNorm=1<br>Divergiu: 0 | La circularité est constante dans l'entraînement ($\sigma_2=0$) ; selon la règle du point 6, $z_2=0$ pour tous les échantillons, et la classification ne dépend que de la superficie dans les deux cas. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0702" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0702 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0702 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0702 button:hover { background: #e8dfcf; }
  #sim-ep0702 button.sim-ep0702_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0702_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0702_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP07_02 : Normalisation Z-score et distance k-NN</span>
  <span class="sim-ep0702_pill">Normalisation des caractéristiques</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição e Seleção de Modo -->
  <div class="sim-ep0702_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:8px;">
      Chaque exemple possède deux caractéristiques : aire (px) et circularité [0, 1]. Basculez la normalisation et observez le changement dans la classe prédite.
    </div>
    
    <div id="sim-ep0702_query" style="font-size:11px; color:#26241d; text-align:center; font-family:monospace; font-weight:700; margin-bottom:10px;"></div>

    <div style="display:flex; justify-content:center; gap:8px; flex-wrap:wrap;">
      <button id="sim-ep0702_btn_raw" class="sim-ep0702_active">Sans normalisation</button>
      <button id="sim-ep0702_btn_norm">Avec normalisation (Z-score)</button>
    </div>
  </div>

  <!-- Cards de Amostras -->
  <div id="sim-ep0702_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0702_debug" class="sim05_ep01_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep02(root){
    if (!root || root.dataset.sim07Ep02Init) return;
    root.dataset.sim07Ep02Init = "1";

    var pontos = [
      {nome: "P1", x1: 10,  x2: 0.90, r: 0},
      {nome: "P2", x1: 12,  x2: 0.85, r: 0},
      {nome: "P3", x1: 900, x2: 0.20, r: 1},
      {nome: "P4", x1: 950, x2: 0.25, r: 1}
    ];

    pontos.forEach(function(p, i){ p.idx = i; });
    var query = {x1: 500, x2: 0.88};
    var k = 3;

    function stats(vals){
      var m = vals.reduce(function(a, b){ return a + b; }, 0) / vals.length;
      var v = vals.reduce(function(a, s){ return a + (s - m) * (s - m); }, 0) / vals.length;
      return {mean: m, std: Math.sqrt(v)};
    }

    var s1 = stats(pontos.map(function(p){ return p.x1; }));
    var s2 = stats(pontos.map(function(p){ return p.x2; }));

    function z(x, s){ return s.std === 0 ? 0 : (x - s.mean) / s.std; }

    var cards   = root.querySelector('#sim-ep0702_cards');
    var dbg     = root.querySelector('#sim-ep0702_debug');
    var qEl     = root.querySelector('#sim-ep0702_query');
    var btnRaw  = root.querySelector('#sim-ep0702_btn_raw');
    var btnNorm = root.querySelector('#sim-ep0702_btn_norm');
    var modoNorm = false;

    function render(){
      btnRaw.classList.toggle('sim-ep0702_active', !modoNorm);
      btnNorm.classList.toggle('sim-ep0702_active', modoNorm);

      qEl.textContent = '★ Requête : Aire = ' + query.x1 + ', Circularidade = ' + query.x2 +
        (modoNorm ? ' → z_área = ' + z(query.x1, s1).toFixed(3) + ', z_circ = ' + z(query.x2, s2).toFixed(3) : '');

      var qx1 = modoNorm ? z(query.x1, s1) : query.x1;
      var qx2 = modoNorm ? z(query.x2, s2) : query.x2;

      var lista = pontos.map(function(p){
        var px1 = modoNorm ? z(p.x1, s1) : p.x1;
        var px2 = modoNorm ? z(p.x2, s2) : p.x2;
        var d = Math.sqrt((px1 - qx1) * (px1 - qx1) + (px2 - qx2) * (px2 - qx2));
        return {nome: p.nome, r: p.r, d: d, idx: p.idx, area: p.x1, circ: p.x2, va: px1, vc: px2};
      });

      lista.sort(function(a, b){ return (a.d - b.d) || (a.idx - b.idx); });

      cards.innerHTML = '';
      var votos = [0, 0];

      lista.forEach(function(p, i){
        var dentro = i < k;
        if (dentro) votos[p.r]++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? (p.r === 0 
                ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
                : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;') 
            : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;');

        var valorUsado = modoNorm
          ? ('z = (' + p.va.toFixed(2) + ', ' + p.vc.toFixed(2) + ')')
          : ('área = ' + p.area + ', circ = ' + p.circ);

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + p.nome + ' (r = ' + p.r + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">' + valorUsado + '</div>' +
          '<div style="font-family:monospace; margin-bottom:4px;">d = ' + p.d.toFixed(3) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'VOTA' : '&ndash;') + '</div>';

        cards.appendChild(div);
      });

      var previsto = votos[1] > votos[0] ? 1 : (votos[0] > votos[1] ? 0 : lista[0].r);

      if (previsto === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = (modoNorm ? 'COM Normalização' : 'SEM Normalização') +
        '  |  k = ' + k + '  |  Votos Classe 0: ' + votos[0] + ', Classe 1: ' + votos[1] +
        '  |  Classe prevista: ' + previsto;
    }

    btnRaw.addEventListener('click', function(){ modoNorm = false; render(); });
    btnNorm.addEventListener('click', function(){ modoNorm = true; render(); });
    render();
  }

  function tryInitSim07Ep02(){
    var root = document.getElementById('sim-ep0702');
    if (root) initSim07Ep02(root); else setTimeout(tryInitSim07Ep02, 200);
  }
  tryInitSim07Ep02();
})();
</script>
""")

**Figure 7.2:** Simulateur EP07_02: Effet de la Normalisation *Z-score* sur la Distance k-NN


In [ ]:
%%writefile EP07_02.py
# Code Python

In [ ]:
TestSuite("EP07_02.py").run()

### EP07_03 🟡 Évaluation par Matrice de Confusion

Un classifieur binaire de qualité de soudure a été entraîné et testé sur une ligne de production. Pour chaque pièce inspectée, le système a enregistré le label **réel** (obtenu par un expert) et le label **prédit** par le classifieur, où `1` représente « défectueuse » et `0` représente « conforme ».

La direction qualité souhaite connaître non seulement l’exactitude du système, mais aussi sa **précision** (lorsque le système signale un défaut, à quelle fréquence a-t-il raison ?) et son **rappel** (parmi toutes les pièces réellement défectueuses, combien le système a-t-il réussi à identifier ?) — la distinction discutée dans la section d’évaluation des classifieurs du chapitre.

#### 📋 Directives d’Implémentation

1. **Quantité :** Lire l’entier $N$ (nombre de pièces inspectées).
2. **Données de chaque pièce :** Pour chacune des $N$ pièces, lire deux entiers — le label réel $y$ et le label prédit $\hat y$ (tous deux $\in \{0, 1\}$).
3. **Matrice de confusion :** En considérant la classe `1` (défectueuse) comme **positive**, compter :
   - $VP$ (Vrai Positif) : $y=1$ et $\hat y=1$ ;
   - $FP$ (Faux Positif) : $y=0$ et $\hat y=1$ ;
   - $FN$ (Faux Négatif) : $y=1$ et $\hat y=0$ ;
   - $VN$ (Vrai Négatif) : $y=0$ et $\hat y=0$.
4. **Métriques :** Calculer
$$
\text{Exactitude} = \frac{VP+VN}{N}, \quad
\text{Précision} = \frac{VP}{VP+FP}, \quad
\text{Rappel} = \frac{VP}{VP+FN}.
$$
5. **Cas dégénérés :** Si $VP+FP=0$ (aucune prédiction positive), afficher `Precisao: indefinida`. Si $VP+FN=0$ (aucun cas positif réel), afficher `Revocacao: indefinida`.
6. **Arrondi :** Toutes les métriques numériques doivent être arrondies à 4 décimales (*round half away from zero*) uniquement lors de l’affichage.

#### 📌 Contraintes Computationnelles

* **Convention de classe positive fixe :** la classe `1` est toujours la classe positive dans cet exercice, indépendamment de sa fréquence relative.
* **Protection contre la division par zéro :** implémentez les cas dégénérés du point 5 avant d’effectuer la division.
* **Ordre de sortie :** suivez exactement l’ordre spécifié dans la section de sortie, même dans les cas dégénérés.

#### 🧠 Fondement Théorique

| Métrique | Question à laquelle elle répond | Sensible au déséquilibre ? |
|---|---|---|
| Exactitude | Quelle fraction des pièces a été classée correctement ? | Oui — peut masquer des erreurs dans la classe minoritaire |
| Précision | Parmi les pièces signalées comme défectueuses, combien le sont réellement ? | Pénalise les faux positifs |
| Rappel | Parmi les pièces réellement défectueuses, combien ont été détectées ? | Pénalise les faux négatifs |

Dans un contexte industriel, un **rappel** faible est souvent plus grave qu’une **précision** faible : laisser passer une pièce défectueuse (faux négatif) tend à être plus coûteux que d’inspecter manuellement une bonne pièce signalée par erreur (faux positif).

#### 📦 Spécification d’Entrée et de Sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $N$.
* Les $N$ lignes suivantes : deux entiers par ligne — $y$ et $\hat y$, séparés par un espace.

**Sortie (dans cet ordre exact) :**

```
VP=<int> FP=<int> FN=<int> VN=<int>
Acuracia: <valeur ou métrique indéfinie>
Precisao: <valeur ou indéfinie>
Revocacao: <valeur ou indéfinie>
```

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 4<br>1 1<br>0 1<br>1 0<br>0 0 | VP=1 FP=1 FN=1 VN=1<br>Acuracia: 0.5000<br>Precisao: 0.5000<br>Revocacao: 0.5000 | Une erreur de chaque type. |
| 3<br>0 0<br>0 0<br>0 0 | VP=0 FP=0 FN=0 VN=3<br>Acuracia: 1.0000<br>Precisao: indefinida<br>Revocacao: indefinida | Aucun cas positif réel ni prédit. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0703" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0703 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0703 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0703 button:hover { background: #e8dfcf; }
  #sim-ep0703 button.sim-ep0703_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0703_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0703_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP07_03 : Précision x Rappel</span>
  <span class="sim-ep0703_pill">Ligne de production</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Seleção de Cenário -->
  <div class="sim-ep0703_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Choisissez un scénario d'inspection et observez comment l'exactitude, la précision et le rappel réagissent différemment.
    </div>

    <div style="display:flex; gap:6px; flex-wrap:wrap; justify-content:center;">
      <button id="sim-ep0703_b1" class="sim-ep0703_active">Scénario A : Erreurs équilibrées</button>
      <button id="sim-ep0703_b2">Scénario B : Faux négatifs</button>
      <button id="sim-ep0703_b3">Scénario C : Aucun défaut réel</button>
      <button id="sim-ep0703_b4">Scénario D : Faux positifs</button>
    </div>
  </div>

  <!-- Cards das Peças do Cenário -->
  <div id="sim-ep0703_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0703_debug" class="sim-ep0703_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep03(root){
    if (!root || root.dataset.sim07Ep03Init) return;
    root.dataset.sim07Ep03Init = "1";

    var cenarios = {
      A: [{y:1, p:1}, {y:0, p:1}, {y:1, p:0}, {y:0, p:0}],
      B: [{y:1, p:0}, {y:1, p:0}, {y:1, p:1}, {y:0, p:0}],
      C: [{y:0, p:0}, {y:0, p:0}, {y:0, p:0}],
      D: [{y:0, p:1}, {y:0, p:1}, {y:1, p:1}, {y:0, p:0}]
    };

    var cards = root.querySelector('#sim-ep0703_cards');
    var dbg   = root.querySelector('#sim-ep0703_debug');

    var botoes = {
      A: root.querySelector('#sim-ep0703_b1'),
      B: root.querySelector('#sim-ep0703_b2'),
      C: root.querySelector('#sim-ep0703_b3'),
      D: root.querySelector('#sim-ep0703_b4')
    };

    function render(key){
      Object.keys(botoes).forEach(function(k){
        botoes[k].classList.toggle('sim-ep0703_active', k === key);
      });

      var dados = cenarios[key];
      var VP = 0, FP = 0, FN = 0, VN = 0;
      cards.innerHTML = '';

      dados.forEach(function(d, i){
        if (d.y === 1 && d.p === 1) VP++;
        else if (d.y === 0 && d.p === 1) FP++;
        else if (d.y === 1 && d.p === 0) FN++;
        else VN++;

        var statusCor = '';
        var statusTxt = '';

        if (d.y === d.p) {
          statusCor = 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;';
          statusTxt = d.y === 1 ? 'VP (Acerto)' : 'VN (Acerto)';
        } else {
          statusCor = 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;';
          statusTxt = d.p === 1 ? 'FP (Alarme Falso)' : 'FN (Escapou)';
        }

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' + statusCor;

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">Peça ' + (i + 1) + '</div>' +
          '<div style="font-family:monospace; font-size:10px; margin-bottom:4px;">Real = ' + d.y + ' | Prev = ' + d.p + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + statusTxt + '</div>';

        cards.appendChild(div);
      });

      var N = dados.length;
      var acc = ((VP + VN) / N).toFixed(4);
      var prec = (VP + FP) > 0 ? (VP / (VP + FP)).toFixed(4) : 'Indefinida';
      var rev = (VP + FN) > 0 ? (VP / (VP + FN)).toFixed(4) : 'Indefinida';

      if (prec === 'Indefinida' || parseFloat(prec) < 0.5) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'VP = ' + VP + ' | FP = ' + FP + ' | FN = ' + FN + ' | VN = ' + VN +
        '  |  Acurácia = ' + acc + '  |  Precisão = ' + prec + '  |  Revocação = ' + rev;
    }

    botoes.A.addEventListener('click', function(){ render('A'); });
    botoes.B.addEventListener('click', function(){ render('B'); });
    botoes.C.addEventListener('click', function(){ render('C'); });
    botoes.D.addEventListener('click', function(){ render('D'); });

    render('A');
  }

  function tryInitSim07Ep03(){
    var root = document.getElementById('sim-ep0703');
    if (root) initSim07Ep03(root); else setTimeout(tryInitSim07Ep03, 200);
  }
  tryInitSim07Ep03();
})();
</script>
""")

**Figure 7.3:** Simulateur EP07_03 : Précision x Rappel


In [ ]:
%%writefile EP07_03.py
# Code Python

In [ ]:
TestSuite("EP07_03.py").run()

### EP07_04 🟠 Codage Manuel du Descripteur LBP

La fonction `local_binary_pattern` de `scikit-image`, utilisée dans le projet de classification de textures, calcule automatiquement le code LBP de chaque pixel d'une image. Avant de l'utiliser comme une boîte noire, vous avez été chargé d'implémenter manuellement le calcul du code LBP classique ($P=8$, $R=1$) pour le pixel central d'un voisinage $3\times3$, exactement comme défini dans l'équation du chapitre.

En plus du code, le système d'inspection de textures doit également savoir si ce motif est **uniforme** — un motif est uniforme lorsque le nombre de transitions ($0\to1$ ou $1\to0$) en parcourant les 8 bits **circulairement** (en revenant du dernier bit au premier) est **au plus 2**, propriété exploitée par la variante *uniforme* du LBP mentionnée dans le chapitre.

#### 📋 Directives d'Implémentation

1. **Quantité :** Lire l'entier $T$ (nombre de voisinages à traiter).
2. **Données de chaque voisinage :** Pour chacun des $T$ voisinages, lire une matrice $3\times3$ d'entiers (intensités), fournie en 3 lignes de 3 valeurs chacune. Le pixel central est la position `[1][1]`.
3. **Ordre des voisins :** Parcourir les 8 voisins dans le sens **horaire**, en commençant par le coin supérieur gauche, dans l'ordre suivant des positions `[ligne][colonne]` : `[0][0]`, `[0][1]`, `[0][2]`, `[1][2]`, `[2][2]`, `[2][1]`, `[2][0]`, `[1][0]`. C'est l'indice $p = 0, 1, \ldots, 7$ de l'équation du LBP.
4. **Fonction seuil :** Pour chaque voisin $p$ avec une intensité $g_p$ et un centre $g_c$, calculer $s(g_p - g_c)$, qui vaut `1` si $g_p \geq g_c$ et `0` sinon.
5. **Code LBP :** Calculer
$$
\mathrm{LBP} = \sum_{p=0}^{7} s(g_p - g_c)\, 2^p.
$$
6. **Transitions :** En considérant la séquence circulaire de bits $s_0, s_1, \ldots, s_7$ (dans l'ordre du point 3), compter combien de paires consécutives **adjacentes dans la séquence circulaire** (y compris la paire $s_7, s_0$) diffèrent entre elles.
7. **Classification :** Si le nombre de transitions est $\le 2$, classer comme `UNIFORME` ; sinon, `NAO_UNIFORME`.
8. **Sortie :** Pour chaque voisinage, dans l'ordre d'entrée, imprimer le code LBP (entier décimal, $0$–$255$), le nombre de transitions et la classification.

#### 📌 Contraintes Computationnelles

* **Ordre fixe des voisins :** l'ordre du point 3 est obligatoire — l'inverser produit un code numériquement différent, même en représentant le même motif visuel.
* **Comparaison non stricte :** $s(z) = 1$ lorsque $z \ge 0$ (le chapitre lui-même définit l'égalité comme incluse dans le cas `1`).
* **Comptage circulaire :** ne pas oublier la paire qui ferme le cycle ($s_7$ avec $s_0$) ; ignorer cette paire est une erreur courante qui classe incorrectement les motifs uniformes.

#### 🧠 Fondement Théorique

| Motif (bits $s_0\ldots s_7$) | Transitions | Interprétation |
|---|---|---|
| `00000000` ou `11111111` | 0 | Région homogène (tache claire ou sombre) |
| `00001111` | 2 | Bord simple entre deux régions |
| `01010101` | 8 | Texture de contraste alterné — non uniforme |

Les motifs uniformes se concentrent dans les régions de texture lisse ou de bords simples ; les motifs non uniformes tendent à correspondre à du bruit haute fréquence. C'est pourquoi l'histogramme LBP *uniforme*, utilisé dans le projet de classification de textures, regroupe tous les motifs non uniformes dans un seul compartiment, réduisant la dimensionnalité du descripteur.

#### 📦 Spécification d'Entrée et de Sortie (VPL)

**Entrée :**

* Ligne 1 : Entier $T$.
* Pour chaque voisinage : 3 lignes avec 3 entiers chacune (matrice $3\times3$).

**Sortie :**

* $T$ lignes, au format `LBP=<int> transicoes=<int> <UNIFORME|NAO_UNIFORME>`.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 1<br>10 10 10<br>10 50 10<br>10 10 10 | LBP=0 transicoes=0 UNIFORME | Le centre est le plus clair ; tous les voisins génèrent un bit 0. |
| 1<br>90 90 90<br>10 50 10<br>90 90 90 | LBP=119 transicoes=4 NAO_UNIFORME | Voisins clairs et sombres alternés dans le voisinage. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0704" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0704 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0704 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0704 button:hover { background: #e8dfcf; }
  #sim-ep0704 button.sim-ep0704_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0704_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0704_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP07_04 : Code LBP d'un voisinage 3&times;3</span>
  <span class="sim-ep0704_pill">P = 8, R = 1</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Seleção de Exemplo -->
  <div class="sim-ep0704_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Cliquez sur une cellule du voisinage pour basculer entre clair et sombre (le centre est fixe) et observez le code LBP résultant. Le libellé p indique l'indice de l'équation.
    </div>

    <div style="display:flex; gap:6px; flex-wrap:wrap; justify-content:center;">
      <button id="sim-ep0704_b1">Exemple 1 : Tache homogène</button>
      <button id="sim-ep0704_b2" class="sim-ep0704_active">Exemple 2 : Motif alterné</button>
    </div>
  </div>

  <!-- Grid Vizinhança 3x3 -->
  <div class="sim-ep0704_panel" style="margin-bottom:14px; text-align:center;">
    <div id="sim-ep0704_grid" style="display:grid; grid-template-columns:repeat(3, 60px); gap:4px; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0704_debug" class="sim-ep0704_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep04(root){
    if (!root || root.dataset.sim07Ep04Init) return;
    root.dataset.sim07Ep04Init = "1";

    var exemplos = {
      1: [[10, 10, 10], [10, 50, 10], [10, 10, 10]],
      2: [[90, 90, 90], [10, 50, 10], [90, 90, 90]]
    };

    var valores = exemplos[2].map(function(row){ return row.slice(); });
    var grid = root.querySelector('#sim-ep0704_grid');
    var dbg  = root.querySelector('#sim-ep0704_debug');
    var btn1 = root.querySelector('#sim-ep0704_b1');
    var btn2 = root.querySelector('#sim-ep0704_b2');

    var ordem = [[0, 0], [0, 1], [0, 2], [1, 2], [2, 2], [2, 1], [2, 0], [1, 0]];
    var pIndex = {};
    ordem.forEach(function(pos, p){ pIndex[pos[0] + ',' + pos[1]] = p; });
    var cenarioAtivo = 2;

    function marcarBotaoAtivo(n){
      cenarioAtivo = n;
      btn1.classList.toggle('sim-ep0704_active', n === 1);
      btn2.classList.toggle('sim-ep0704_active', n === 2);
    }

    function render(){
      grid.innerHTML = '';
      for (var r = 0; r < 3; r++){
        for (var c = 0; c < 3; c++){
          (function(r, c){
            var v = valores[r][c];
            var central = (r === 1 && c === 1);
            var div = document.createElement('div');

            var bordaCor = central ? '#26241d' : '#e4dcc8';
            var textoCor = v > 128 ? '#26241d' : '#ffffff';

            div.style.cssText = 'position:relative; height:60px; display:flex; align-items:center; justify-content:center; font-family:monospace; font-weight:700; border-radius:6px; cursor:' + (central ? 'default' : 'pointer') + '; border:2px solid ' + bordaCor + '; background:rgb(' + v + ',' + v + ',' + v + '); color:' + textoCor + '; transition:all 0.15s ease;';
            div.textContent = v;

            if (!central){
              var pLabel = document.createElement('span');
              pLabel.textContent = 'p' + pIndex[r + ',' + c];
              pLabel.style.cssText = 'position:absolute; top:2px; left:4px; font-size:9px; font-weight:400; opacity:0.8;';
              div.appendChild(pLabel);

              div.addEventListener('click', function(){
                valores[r][c] = valores[r][c] >= 128 ? 10 : 200;
                cenarioAtivo = null;
                btn1.classList.remove('sim-ep0704_active');
                btn2.classList.remove('sim-ep0704_active');
                render();
              });
            }
            grid.appendChild(div);
          })(r, c);
        }
      }

      var gc = valores[1][1];
      var bits = ordem.map(function(pos){ return valores[pos[0]][pos[1]] >= gc ? 1 : 0; });
      var lbp = 0;
      bits.forEach(function(b, p){ lbp += b * Math.pow(2, p); });

      var trans = 0;
      for (var i = 0; i < 8; i++){
        if (bits[i] !== bits[(i + 1) % 8]) trans++;
      }

      var classe = trans <= 2 ? 'UNIFORME' : 'NÃO-UNIFORME';

      if (trans <= 2) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'bits (p0..p7) = ' + bits.join('') + '  |  LBP = ' + lbp + '  |  transições = ' + trans + '  |  ' + classe;
    }

    btn1.addEventListener('click', function(){
      valores = exemplos[1].map(function(row){ return row.slice(); });
      marcarBotaoAtivo(1);
      render();
    });

    btn2.addEventListener('click', function(){
      valores = exemplos[2].map(function(row){ return row.slice(); });
      marcarBotaoAtivo(2);
      render();
    });

    marcarBotaoAtivo(2);
    render();
  }

  function tryInitSim07Ep04(){
    var root = document.getElementById('sim-ep0704');
    if (root) initSim07Ep04(root); else setTimeout(tryInitSim07Ep04, 200);
  }
  tryInitSim07Ep04();
})();
</script>
""")

**Figure 7.4:** Simulateur EP07_04 : Code LBP d


In [ ]:
%%writefile EP07_04.py
# Code Python

In [ ]:
TestSuite("EP07_04.py").run()

### EP07_05 🔴 Histogramme des orientations d’une cellule HOG

La fonction `hog` de `scikit-image`, utilisée dans le projet de classification de chiffres, divise l’image en petites **cellules** et, pour chacune, construit un histogramme des orientations du gradient pondéré par la magnitude — exactement l’étape centrale décrite dans la section sur le descripteur HOG du chapitre.

Vous êtes chargé d’implémenter ce calcul pour une seule cellule, à partir des valeurs de magnitude et d’orientation du gradient **déjà calculées** pour chaque pixel de la cellule (ce qui évite le calcul des dérivées partielles).

#### 📋 Directives d’implémentation

1. **Dimensions :** Lire les entiers $n$ (la cellule a $n \times n$ pixels) et $B$ (nombre de compartiments de l’histogramme).
2. **Magnitudes :** Lire $n$ lignes avec chacune $n$ valeurs réelles, représentant $|\nabla f(x,y)|$ pour chaque pixel de la cellule.
3. **Orientations :** Lire encore $n$ lignes avec chacune $n$ valeurs réelles, représentant $\theta(x,y)$ en **degrés**, déjà converties dans l’intervalle **non signé** $[0^\circ, 180^\circ)$, comme conventionnellement utilisé par le HOG.
4. **Compartiments :** Les $B$ compartiments couvrent $[0^\circ, 180^\circ)$ en bandes égales de largeur $180/B$ degrés. Un pixel avec une orientation $\theta$ appartient au compartiment $\lfloor \theta / (180/B) \rfloor$ ; si cet indice est égal à $B$ (possible uniquement lorsque $\theta$ est exactement $180^\circ$, ce qui ne devrait pas se produire selon la directive du point 3), utilisez le compartiment $B-1$.
5. **Histogramme brut :** Pour chaque pixel, accumulez sa **magnitude** (et non son compte) dans le compartiment correspondant :
$$
H[b] = \sum_{(x,y)\, :\, \text{bin}(\theta(x,y)) = b} |\nabla f(x,y)|.
$$
6. **Normalisation L2 :** Après avoir construit $H$, normalisez-le pour obtenir $\hat H$ :
$$
\hat H[b] = \frac{H[b]}{\sqrt{\sum_{j=0}^{B-1} H[j]^2 + \epsilon}}, \qquad \epsilon = 10^{-6}.
$$
7. **Sortie :** Imprimez l’histogramme brut $H$ (arrondi à 2 décimales) sur une ligne, suivi de l’histogramme normalisé $\hat H$ (arrondi à 4 décimales) sur une autre ligne, tous deux avec les $B$ valeurs séparées par des espaces, dans l’ordre des compartiments.

#### 📌 Contraintes computationnelles

* ***Binning* non signé :** l’intervalle des orientations est $[0,180)$, pas $[0,360)$ — les gradients dans des directions opposées (différence de $180^\circ$) contribuent au **même** compartiment, convention standard du HOG pour la détection d’objets.
* **Accumulation par magnitude, pas par compte :** l’histogramme pondère chaque pixel par sa magnitude de gradient, il ne compte pas simplement combien de pixels tombent dans chaque compartiment.
* **Constante de stabilisation :** le $\epsilon = 10^{-6}$ dans le dénominateur de la normalisation évite la division par zéro lorsque la cellule est complètement homogène (toutes les magnitudes nulles).

#### 📐 D’où proviennent les matrices d’entrée

Avant cet EP, chaque pixel $(x,y)$ de l’image passe par :

$$
G_x = f(x+1,y)-f(x-1,y), \qquad G_y = f(x,y+1)-f(x,y-1)
$$

$$
|\nabla f| = \sqrt{G_x^2+G_y^2}, \qquad \theta_{\text{signé}} = \operatorname{atan2}(G_y,G_x)
$$

Comme le HOG ignore la polarité du contraste, l’angle est replié dans l’intervalle non signé :

$$
\theta = \theta_{\text{signé}} \bmod 180°
$$

En répétant cela pour tous les pixels d’une cellule $n\times n$, on obtient les deux matrices d’entrée de cet exercice : les **magnitudes** $|\nabla f|$ et les **orientations** $\theta \in [0°,180°)$.

#### 🧠 Fondement théorique

| Étape | Rôle |
|---|---|
| Magnitude du gradient | Pondère la contribution de chaque pixel — les bords forts pèsent plus que le bruit faible |
| Orientation non signée | Rend le descripteur invariant à la polarité du contraste (clair→sombre vs. sombre→clair) |
| Histogramme par cellule | Résume la distribution locale des bords en un vecteur compact |
| Normalisation L2 | Réduit la sensibilité du descripteur aux variations globales d’éclairage et de contraste |

La concaténation des histogrammes normalisés de toutes les cellules de l’image — non implémentée dans cet exercice — forme le vecteur de caractéristiques HOG complet, utilisé comme entrée du classifieur k-NN dans le projet du chapitre.

#### 📦 Spécification d’entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : entiers $n$ et $B$.
* Les $n$ lignes suivantes : $n$ magnitudes réelles chacune.
* Les $n$ lignes suivantes : $n$ orientations réelles (degrés, $[0,180)$) chacune.

**Sortie :**

* Ligne 1 : les $B$ valeurs de l’histogramme brut, arrondies à 2 décimales.
* Ligne 2 : les $B$ valeurs de l’histogramme normalisé, arrondies à 4 décimales.

#### 📌 Exemples

| Entrée | Sortie | Observation |
|---|---|---|
| 2 2<br>1.0 2.0<br>3.0 4.0<br>10 100<br>170 20 | 5.00 5.00<br>0.7071 0.7071 | Bin de largeur 90° : $[0,90)$ et $[90,180)$ ; les magnitudes 1 et 4 tombent dans le bin 0, 2 et 3 dans le bin 1. |
| 2 4<br>0.0 0.0<br>0.0 0.0<br>0 0<br>0 0 | 0.00 0.00 0.00 0.00<br>0.0000 0.0000 0.0000 0.0000 | Cellule homogène : $\epsilon$ évite la division par zéro. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0705" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">  
<style>
  #sim-ep0705 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0705 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0705 button:hover { background: #e8dfcf; }
  #sim-ep0705 button.sim-ep0705_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0705_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0705_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP07_05 : Histogramme d'orientations d'une cellule</span>
  <span class="sim-ep0705_pill">🔴 cellule 3×3 fixe</span>
</div>

  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Ajustez B et suivez comment la <b>matrice d'orientations</b> (indépendante de celle des magnitudes) est mappée
      vers les compartiments via <code>bin = floor(θ / (180/B))</code>, et comment les magnitudes sont additionnées dans chaque bin.
    </p>

    <!-- Controle B -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Nombre de compartiments (B)</label>
        <span id="ep0705_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">2</span>
      </div>
      <input id="ep0705_sl" style="width:100%;accent-color:#2980b9;" max="6" min="2" step="1" type="range" value="2">
    </div>

    <!-- Entrada bruta (formato VPL) -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📄 Entrée (exactement comme le programme la lit via stdin)</div>
      <pre id="ep0705_stdin" style="background:#1e1e1e;color:#d4d4d4;border-radius:8px;padding:12px 14px;font-size:12px;line-height:1.5;overflow-x:auto;margin:0;"></pre>
    </div>

    <!-- Duas matrizes separadas -->
    <div style="display:flex;gap:16px;flex-wrap:wrap;margin-bottom:20px;">
      <div style="flex:1;min-width:220px;">
        <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🔢 Matrice des magnitudes |∇f|</div>
        <div id="ep0705_mag_grid" style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px;"></div>
      </div>
      <div style="flex:1;min-width:220px;">
        <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📐 Matrice des orientations θ (degrés) — colorée par bin</div>
        <div id="ep0705_ang_grid" style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px;"></div>
      </div>
    </div>

    <!-- Regua 0-180 -->
    <div style="margin-bottom:22px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:10px;">📏 Où chaque θ tombe sur la règle [0°, 180°) — <code>bin = floor(θ / largeur)</code></div>
      <div style="position:relative;height:70px;margin:0 6px;">
        <div id="ep0705_regua" style="position:absolute;top:28px;left:0;right:0;height:14px;border-radius:7px;overflow:hidden;display:flex;border:1px solid #d1d5db;"></div>
        <div id="ep0705_regua_ticks" style="position:absolute;top:44px;left:0;right:0;height:14px;"></div>
        <div id="ep0705_regua_marcas" style="position:absolute;top:0;left:0;right:0;height:26px;"></div>
      </div>
    </div>

    <!-- Faixas dos compartimentos -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📊 Plages de chaque compartiment (largeur = 180° / B)</div>
      <div id="ep0705_faixas" style="display:flex;flex-wrap:wrap;gap:6px;"></div>
    </div>

    <!-- Grade de pixels colorida por bin (mag + ang juntos) -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🧩 Chaque pixel : magnitude + orientation → bin</div>
      <div id="ep0705_pixels" style="display:grid;grid-template-columns:repeat(3,1fr);gap:8px;"></div>
    </div>

    <!-- Botões -->
    <div style="display:flex;gap:8px;justify-content:center;margin-bottom:14px;">
      <button id="ep0705_btn_raw" class="ep0705_btn">Histogramme brut (H)</button>
      <button id="ep0705_btn_norm" class="ep0705_btn">Histogramme normalisé (Ĥ)</button>
    </div>

    <!-- Barras -->
    <div id="ep0705_bars" style="display:flex;gap:6px;align-items:flex-end;height:120px;justify-content:center;margin-bottom:14px;"></div>

    <!-- Passo a passo -->
    <div style="margin-bottom:6px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🧮 Calcul pas à pas (division entière + somme des magnitudes par bin)</div>
      <div id="ep0705_passos" style="background:#f3f4f6;border-radius:8px;padding:10px 12px;font-family:monospace;font-size:11px;color:#374151;line-height:1.8;"></div>
    </div>

    <div id="ep0705_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;margin-top:12px;"></div>
  </div>
</div>
<style>
  #sim-ep0705 .ep0705_btn { font-size:11px;padding:6px 10px;border-radius:6px;border:1px solid #ddd;background:#fff;cursor:pointer; }
  #sim-ep0705 .ep0705_btn.ativo { background:#2980b9;color:#fff;border-color:#2980b9; }
</style>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var n = 3;
    var mags = [[1.0,2.0,0.5],[3.0,4.0,1.5],[0.8,2.5,3.2]];
    var angs = [[10,100,45],[170,20,95],[60,150,5]];
    var CORES = ["#6366f1","#0ea5e9","#10b981","#f59e0b","#ef4444","#a855f7"];

    var slEl = root.querySelector("#ep0705_sl");
    var vlEl = root.querySelector("#ep0705_vl");
    var stdinEl = root.querySelector("#ep0705_stdin");
    var magGridEl = root.querySelector("#ep0705_mag_grid");
    var angGridEl = root.querySelector("#ep0705_ang_grid");
    var reguaEl = root.querySelector("#ep0705_regua");
    var reguaTicksEl = root.querySelector("#ep0705_regua_ticks");
    var reguaMarcasEl = root.querySelector("#ep0705_regua_marcas");
    var faixasEl = root.querySelector("#ep0705_faixas");
    var pxEl = root.querySelector("#ep0705_pixels");
    var bars = root.querySelector("#ep0705_bars");
    var passosEl = root.querySelector("#ep0705_passos");
    var dbg = root.querySelector("#ep0705_debug");
    var btnRaw = root.querySelector("#ep0705_btn_raw");
    var btnNorm = root.querySelector("#ep0705_btn_norm");
    var modoNorm = false;

    function render(){
      btnRaw.classList.toggle("ativo", !modoNorm);
      btnNorm.classList.toggle("ativo", modoNorm);

      var B = parseInt(slEl.value);
      vlEl.textContent = B;
      var largura = 180/B;

      // ---- Entrada bruta (stdin) ----
      var linhas = [];
      linhas.push(n + " " + B);
      mags.forEach(function(row){ linhas.push(row.map(function(v){return v.toFixed(1);}).join(" ")); });
      angs.forEach(function(row){ linhas.push(row.join(" ")); });
      stdinEl.textContent = linhas.join("\\n");

      // ---- bin de cada pixel (floor(theta/largura), clip) ----
      var binsMat = [];
      for(var i=0;i<n;i++){
        binsMat.push([]);
        for(var j=0;j<n;j++){
          var raw = angs[i][j]/largura;
          var b = Math.floor(raw);
          if(b > B-1) b = B-1;
          if(b < 0) b = 0;
          binsMat[i].push(b);
        }
      }

      // ---- Matriz de magnitudes (grid simples) ----
      magGridEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var d = document.createElement("div");
          d.style.cssText = "text-align:center;border-radius:8px;padding:8px 4px;font-size:12px;font-family:monospace;background:#f9fafb;border:1px solid #e5e7eb;color:#374151;";
          d.textContent = mags[i][j].toFixed(1);
          magGridEl.appendChild(d);
        }
      }

      // ---- Matriz de orientações (colorida por bin, com floor explícito) ----
      angGridEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var b2 = binsMat[i][j];
          var cor2 = CORES[b2];
          var raw2 = angs[i][j]/largura;
          var d2 = document.createElement("div");
          d2.style.cssText = "text-align:center;border-radius:8px;padding:6px 4px;font-size:11px;font-family:monospace;background:"+cor2+"22;border:2px solid "+cor2+";color:#374151;";
          d2.innerHTML = "<div style=\\"font-weight:700;\\">"+angs[i][j]+"°</div>"+
            "<div style=\\"font-size:9px;color:#6b7280;\\">÷"+largura.toFixed(1)+"="+raw2.toFixed(2)+"</div>"+
            "<div style=\\"font-size:9px;font-weight:700;color:"+cor2+";\\">⌊·⌋=bin "+b2+"</div>";
          angGridEl.appendChild(d2);
        }
      }

      // ---- Régua 0-180 com faixas coloridas ----
      reguaEl.innerHTML = "";
      for(var b3=0;b3<B;b3++){
        var seg = document.createElement("div");
        seg.style.cssText = "flex:1;background:"+CORES[b3]+";opacity:0.35;border-right:1px solid rgba(255,255,255,0.6);";
        reguaEl.appendChild(seg);
      }
      // ticks (limites dos bins)
      reguaTicksEl.innerHTML = "";
      for(var b4=0;b4<=B;b4++){
        var pct = (b4*largura/180*100);
        var tick = document.createElement("div");
        tick.style.cssText = "position:absolute;left:"+pct+"%;top:0;font-size:9px;color:#6b7280;transform:translateX(-50%);white-space:nowrap;";
        tick.textContent = (b4*largura).toFixed(0)+"°";
        reguaTicksEl.appendChild(tick);
      }
      // marcadores dos angulos de cada pixel
      reguaMarcasEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var ang = angs[i][j];
          var b5 = binsMat[i][j];
          var pctm = (ang/180*100);
          var marker = document.createElement("div");
          marker.style.cssText = "position:absolute;left:"+pctm+"%;top:0;transform:translateX(-50%);display:flex;flex-direction:column;align-items:center;";
          marker.innerHTML = "<div style=\\"font-size:9px;color:"+CORES[b5]+";font-weight:700;\\">("+i+","+j+")</div>"+
            "<div style=\\"width:0;height:0;border-left:5px solid transparent;border-right:5px solid transparent;border-top:8px solid "+CORES[b5]+";\\"></div>";
          reguaMarcasEl.appendChild(marker);
        }
      }

      // ---- Faixas dos bins (legenda) ----
      faixasEl.innerHTML = "";
      for(var b=0;b<B;b++){
        var lo = (b*largura).toFixed(1);
        var hi = ((b+1)*largura).toFixed(1);
        var chip = document.createElement("div");
        chip.style.cssText = "display:flex;align-items:center;gap:6px;background:#f9fafb;border:1px solid #e5e7eb;border-radius:20px;padding:4px 10px;font-size:11px;color:#374151;";
        chip.innerHTML = "<span style=\\"width:10px;height:10px;border-radius:50%;background:"+CORES[b]+";display:inline-block;\\"></span>bin "+b+": ["+lo+"°, "+hi+"°)";
        faixasEl.appendChild(chip);
      }

      // ---- Atribuição por pixel + histograma bruto ----
      var H = new Array(B).fill(0);
      var binsPorPixel = [];
      pxEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var mag = mags[i][j], ang = angs[i][j];
          var bin = binsMat[i][j];
          binsPorPixel.push({i:i, j:j, mag:mag, ang:ang, bin:bin});
          H[bin] += mag;

          var div = document.createElement("div");
          var cor = CORES[bin];
          div.style.cssText = "text-align:center;border-radius:10px;padding:8px 6px;font-size:11px;background:"+cor+"22;border:2px solid "+cor+";color:#374151;";
          div.innerHTML = "<div style=\\"font-weight:700;\\">mag="+mag.toFixed(1)+"</div>"+
            "<div style=\\"font-family:monospace;\\">θ="+ang+"°</div>"+
            "<div style=\\"font-weight:700;color:"+cor+";\\">→ bin "+bin+"</div>";
          pxEl.appendChild(div);
        }
      }

      var denom = Math.sqrt(H.reduce(function(s,v){return s+v*v;},0) + 1e-6);
      var Hn = H.map(function(v){ return v/denom; });

      // ---- Barras (coloridas por bin) ----
      var dados = modoNorm ? Hn : H;
      var maxD = Math.max.apply(null, dados.concat([0.001]));
      bars.innerHTML = "";
      dados.forEach(function(v, b){
        var col = document.createElement("div");
        col.style.cssText = "display:flex;flex-direction:column;align-items:center;gap:4px;";
        var barra = document.createElement("div");
        var altura = Math.round((v/maxD)*90) + 4;
        barra.style.cssText = "width:34px;height:"+altura+"px;background:"+CORES[b]+";border-radius:4px 4px 0 0;";
        var label = document.createElement("div");
        label.style.cssText = "font-family:monospace;font-size:10px;color:#4b5563;";
        label.textContent = modoNorm ? v.toFixed(4) : v.toFixed(2);
        var binLabel = document.createElement("div");
        binLabel.style.cssText = "font-size:9px;color:#9ca3af;";
        binLabel.textContent = "bin "+b;
        col.appendChild(barra);
        col.appendChild(label);
        col.appendChild(binLabel);
        bars.appendChild(col);
      });

      // ---- Passo a passo (floor + soma) ----
      var passos = [];
      for(var b=0;b<B;b++){
        var contribs = binsPorPixel.filter(function(p){ return p.bin===b; });
        var termos = contribs.map(function(p){ return p.mag.toFixed(2)+" (θ="+p.ang+"°→⌊"+(p.ang/largura).toFixed(2)+"⌋="+p.bin+")"; }).join(" + ");
        if(termos === "") termos = "(nenhum pixel)";
        passos.push("<span style=\\"color:"+CORES[b]+";font-weight:700;\\">H["+b+"]</span> = "+termos+" = <b>"+H[b].toFixed(2)+"</b>");
      }
      passosEl.innerHTML = passos.join("<br>");

      dbg.textContent = "H=[" + H.map(function(v){return v.toFixed(2);}).join(", ") + "]  |  Ĥ=[" +
        Hn.map(function(v){return v.toFixed(4);}).join(", ") + "]";
    }

    slEl.addEventListener("input", render);
    btnRaw.addEventListener("click", function(){ modoNorm = false; render(); });
    btnNorm.addEventListener("click", function(){ modoNorm = true; render(); });
    render();
  }
  function tryInit(){
    var root = document.getElementById("sim-ep0705");
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 7.5:** Simulateur EP07_05 : Histogramme HOG d


In [ ]:
%%writefile EP07_05.py
# Code Python

In [ ]:
TestSuite("EP07_05.py").run()

### EP07_06 🟣 *Pipeline* complet : Descripteurs + k-NN + Évaluation multi-classe

Cet exercice intègre les trois étapes centrales du chapitre dans un *pipeline* unique, reproduisant en miniature le **Projet pratique 2** (classification de textures synthétiques par LBP) : un ensemble d'histogrammes de descripteurs **déjà extraits** (comme s'il s'agissait d'histogrammes LBP) est utilisé pour entraîner un classifieur k-NN, qui est à son tour évalué sur un ensemble de test indépendant au moyen d'une matrice de confusion multi-classe.

Contrairement à l'EP07_01, ici l'espace des caractéristiques a une dimension arbitraire $H$ (la taille de l'histogramme), il existe plus de deux classes, et la métrique de distance est un paramètre d'entrée — ce qui permet de reproduire l'expérience de comparaison de métriques discutée dans le chapitre.

#### 📋 Directives d'implémentation

1. **Classes :** Lire l'entier $C$ (nombre de classes) suivi de $C$ noms de classe (*strings* sans espace), dans l'ordre où ils doivent apparaître dans la matrice de confusion.
2. **Configuration :** Lire l'entier $H$ (dimension des histogrammes), la *string* $M$ (métrique : `euclidiana` ou `manhattan`) et l'entier impair $k$.
3. **Entraînement :** Lire l'entier $N$ puis $N$ lignes, chacune contenant le nom de la classe suivi de $H$ valeurs réelles (l'histogramme du descripteur).
4. **Test :** Lire l'entier $Q$ puis $Q$ lignes, chacune contenant le nom de la classe **réelle** suivi de $H$ valeurs réelles (l'histogramme du descripteur de l'échantillon de test).
5. **Distance :** Pour chaque échantillon de test, calculer la distance à chaque exemple d'entraînement en utilisant la métrique $M$ :
$$
d_{\text{euclidiana}}(u,v) = \sqrt{\sum_{j=1}^{H}(u_j-v_j)^2}, \qquad
d_{\text{manhattan}}(u,v) = \sum_{j=1}^{H} |u_j - v_j|.
$$
6. **Classification k-NN :** Sélectionner les $k$ exemples d'entraînement les plus proches (départage des distances par l'ordre de lecture, comme dans l'EP07_01) et classer par la classe majoritaire parmi eux. En cas **d'égalité de vote** entre deux ou plusieurs classes, choisir celle qui apparaît **en premier** dans la liste des classes du point 1.
7. **Matrice de confusion :** Construire une matrice $C \times C$ où la ligne correspond à la classe réelle et la colonne à la classe prédite, en suivant l'ordre des classes du point 1.
8. **Précision :** Calculer la précision globale comme le rapport entre les succès et $Q$.
9. **Sortie :** Pour chaque échantillon de test, dans l'ordre d'entrée, imprimer la classe prédite. Ensuite, imprimer la matrice de confusion (une ligne par classe réelle, valeurs séparées par des espaces, dans l'ordre des classes). Enfin, imprimer la précision arrondie à 4 décimales.

#### 📌 Contraintes computationnelles

* **Métrique sélectionnable :** implémenter les deux distances ; la métrique $M$ définit celle utilisée pour toute l'exécution (il n'est pas possible de mélanger les métriques dans le même appel).
* **Départage de vote déterministe :** le critère du point 6 (ordre de la liste des classes) doit être suivi même lorsque l'égalité implique plus de deux classes.
* **Indépendance de l'entraînement et du test :** il n'est pas nécessaire de vérifier que les échantillons de test n'apparaissent pas dans l'entraînement — supposer que l'entrée est valide.

#### 🧠 Fondement théorique

| Étape de l'exercice | Étape correspondante dans le chapitre |
|---|---|
| Histogrammes d'entraînement/test déjà extraits | `descritor_lbp` appliqué aux textures synthétiques |
| Distance euclidienne ou Manhattan | Paramètre `metric` du `KNeighborsClassifier` |
| Vote majoritaire avec $k$ voisins | `KNeighborsClassifier.predict` |
| Matrice de confusion $C\times C$ | `confusion_matrix` de `scikit-learn` |
| Précision globale | `accuracy_score` de `scikit-learn` |

Cet exercice met en évidence, de manière contrôlée, un résultat discuté dans le chapitre : le **choix de la métrique de distance** et de la **valeur de $k$** peut modifier la classe prédite pour un même échantillon, même en maintenant fixe le descripteur utilisé — renforçant que, dans la reconnaissance de formes classique, le descripteur, la métrique et le classifieur forment un système interdépendant, et non des pièces isolées.

#### 📦 Spécification d'entrée et de sortie (VPL)

**Entrée :**

* Ligne 1 : entier $C$ suivi de $C$ noms de classe.
* Ligne 2 : entier $H$, *string* $M$ et entier $k$.
* Ligne 3 : entier $N$.
* Prochaines $N$ lignes d'entraînement : nom de la classe suivi de $H$ réels.
* Ligne suivante : entier $Q$.
* Prochaines $Q$ lignes de test : nom de la classe réelle suivi de $H$ réels.

**Sortie :**

* $Q$ lignes avec la classe prédite de chaque échantillon de test, dans l'ordre d'entrée.
* $C$ lignes avec la matrice de confusion (une ligne par classe réelle).
* Dernière ligne : `Acuracia: <valeur>`.

#### 📌 Exemples

| Entrée (résumée) | Sortie | Observation |
|---|---|---|
| 2 granular listrada<br>2 euclidiana 1<br>4<br>granular 0.9 0.1<br>granular 0.8 0.2<br>listrada 0.1 0.9<br>listrada 0.2 0.8<br>2<br>granular 0.85 0.15<br>listrada 0.15 0.85 | granular<br>listrada<br>1 0<br>0 1<br>Acuracia: 1.0000 | Avec $k=1$, chaque test est classé par le voisin d'entraînement le plus proche. |

> ### 📝 Nota
>
> Ce simulateur utilise un ensemble simplifié de **3 classes** (`granulaire`, `striée`, `tachetée`) sur des points 2D fictifs, uniquement pour illustrer le *pipeline* de vote, de départage et de matrice de confusion du k-NN. Dans le cadre du **EP07_07**, vous appliquerez cette même logique à une mosaïque d'image réelle, qui introduit une quatrième classe (`échiquier`) et remplace les points 2D par des histogrammes LBP extraits directement des pixels de l'image.

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0706" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">  
<style>
  #sim-ep0706 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0706 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0706 button:hover { background: #e8dfcf; }
  #sim-ep0706 button.sim-ep0706_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0706_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0706_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP07_06 : Pipeline k-NN multi-classe</span>
  <span class="sim-ep0706_pill">6 Entraînement &middot; 3 Test &middot; 3 Classes</span>
</div>

  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Choisissez la métrique, la valeur de k et l'échantillon de test (★). Voyez les k plus proches voisins, le vote,
      le départage si nécessaire, et comment cela se propage à la matrice de confusion et à la précision de l'ensemble entier.
    </p>

    <!-- Controles -->
    <div style="display:flex;flex-wrap:wrap;gap:18px;justify-content:center;margin-bottom:16px;">
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Métrique (M)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_be" class="ep0706_btn">Euclidienne</button>
          <button id="ep0706_bm" class="ep0706_btn">Manhattan</button>
        </div>
      </div>
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Voisins (k)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_k1" class="ep0706_btn">k=1</button>
          <button id="ep0706_k3" class="ep0706_btn">k=3</button>
          <button id="ep0706_k5" class="ep0706_btn">k=5</button>
        </div>
      </div>
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Échantillon de test (★)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_t0" class="ep0706_btn">test 1</button>
          <button id="ep0706_t1" class="ep0706_btn">test 2</button>
          <button id="ep0706_t2" class="ep0706_btn">test 3</button>
        </div>
      </div>
    </div>

    <!-- Legenda -->
    <div id="ep0706_legenda" style="display:flex;gap:10px;justify-content:center;margin-bottom:10px;"></div>

    <!-- Dispersao 2D -->
    <div style="max-width:340px;margin:0 auto 16px auto;height:300px;border:1px solid #e5e7eb;border-radius:12px;background:#fafafa;">
      <div id="ep0706_svg_container" style="width:100%;height:100%;"></div>
    </div>

    <!-- Distancias ordenadas -->
    <div style="margin-bottom:16px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📏 Distances jusqu'à l'échantillon de test (triées) — <span style="font-weight:400;font-size:10px;color:#8a8672;">#i = ordre de lecture dans la liste d'entraînement (survolez)</span></div>
      <div id="ep0706_dists" style="display:grid;grid-template-columns:1fr 1fr;gap:2px 10px;font-family:monospace;font-size:10px;"></div>
    </div>

    <!-- Votacao -->
    <div style="margin-bottom:16px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🗳️ Vote parmi les k voisins</div>
      <div id="ep0706_votos" style="display:flex;gap:10px;justify-content:center;margin-bottom:6px;"></div>
      <div id="ep0706_previsao" style="text-align:center;font-size:12px;font-weight:bold;"></div>
    </div>

    <!-- Matriz de confusao + acuracia (conjunto de teste inteiro) -->
    <div style="margin-bottom:8px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📋 Matrice de confusion et précision — exécution du pipeline sur les 3 échantillons de test</div>
      <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:center;justify-content:center;">
        <table id="ep0706_cm" style="border-collapse:collapse;font-size:11px;font-family:monospace;"></table>
        <div id="ep0706_acc" style="font-size:13px;font-weight:bold;color:#5e5a4a;"></div>
      </div>
    </div>

    <div id="ep0706_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;text-align:center;margin-top:12px;"></div>
  </div>
</div>
<style>
  #sim-ep0706 .ep0706_btn { font-size:11px;padding:5px 10px;border-radius:6px;border:1px solid #ddd;background:#fff;cursor:pointer; }
  #sim-ep0706 .ep0706_btn.ativo { background:#7c3aed;color:#fff;border-color:#7c3aed; }
</style>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var classes = ["granular","listrada","manchada"];
    var CORES = {granular:"#6366f1", listrada:"#f59e0b", manchada:"#10b981"};

    var trainPts = [
      {nome:"granular_1", cls:"granular", x:0.70, y:0.70},
      {nome:"granular_2", cls:"granular", x:0.25, y:0.85},
      {nome:"listrada_1", cls:"listrada", x:0.85, y:0.50},
      {nome:"listrada_2", cls:"listrada", x:0.60, y:0.15},
      {nome:"manchada_1", cls:"manchada", x:0.30, y:0.30},
      {nome:"manchada_2", cls:"manchada", x:0.15, y:0.55}
    ];
    var testPts = [
      {nome:"teste 1", cls:"granular", x:0.50, y:0.50},
      {nome:"teste 2", cls:"listrada", x:0.70, y:0.20},
      {nome:"teste 3", cls:"manchada", x:0.20, y:0.40}
    ];

    var svgContainer = root.querySelector("#ep0706_svg_container");
    var svg = svgNS("svg");
    svg.setAttribute("viewBox", "0 0 100 100");
    svg.setAttribute("style", "width:100%;height:100%;");
    svgContainer.appendChild(svg);
    var legendaEl = root.querySelector("#ep0706_legenda");
    var distsEl = root.querySelector("#ep0706_dists");
    var votosEl = root.querySelector("#ep0706_votos");
    var previsaoEl = root.querySelector("#ep0706_previsao");
    var cmEl = root.querySelector("#ep0706_cm");
    var accEl = root.querySelector("#ep0706_acc");
    var dbg = root.querySelector("#ep0706_debug");

    var be = root.querySelector("#ep0706_be"), bm = root.querySelector("#ep0706_bm");
    var bk1 = root.querySelector("#ep0706_k1"), bk3 = root.querySelector("#ep0706_k3"), bk5 = root.querySelector("#ep0706_k5");
    var bt0 = root.querySelector("#ep0706_t0"), bt1 = root.querySelector("#ep0706_t1"), bt2 = root.querySelector("#ep0706_t2");

    var metrica = "euclidiana", k = 1, testSel = 0;

    function dist(u, v){
      var dx = u.x-v.x, dy = u.y-v.y;
      if(metrica === "euclidiana") return Math.sqrt(dx*dx+dy*dy);
      return Math.abs(dx)+Math.abs(dy);
    }

    function knnPredict(xtest){
      var ds = trainPts.map(function(p, i){ return {i:i, p:p, d:dist(xtest, p)}; });
      ds.sort(function(a,b){ return a.d - b.d; }); // ordem estavel = desempate por ordem de leitura
      var viz = ds.slice(0, k);
      var votos = {}; classes.forEach(function(c){ votos[c]=0; });
      viz.forEach(function(v){ votos[v.p.cls]++; });
      var maxV = Math.max.apply(null, classes.map(function(c){return votos[c];}));
      var empatados = classes.filter(function(c){ return votos[c]===maxV; });
      var pred = empatados[0]; // primeira classe da lista entre as empatadas
      return {pred:pred, viz:viz, votos:votos, empatados:empatados, ordenados:ds};
    }

    function svgNS(tag){
      // Concatenado de propósito: evita que filtros de auto-link do Moodle
      // reconheçam "http://www.w3.org/2000/svg" como URL e insiram uma tag <a>
      // dentro desta string, o que quebraria a sintaxe do createElementNS.
      var SVG_NS = "http" + "://www.w3.org/2000/svg";
      return document.createElementNS(SVG_NS, tag);
    }

    function render(){
      be.classList.toggle("ativo", metrica==="euclidiana");
      bm.classList.toggle("ativo", metrica==="manhattan");
      bk1.classList.toggle("ativo", k===1);
      bk3.classList.toggle("ativo", k===3);
      bk5.classList.toggle("ativo", k===5);
      bt0.classList.toggle("ativo", testSel===0);
      bt1.classList.toggle("ativo", testSel===1);
      bt2.classList.toggle("ativo", testSel===2);

      // Legenda
      legendaEl.innerHTML = "";
      classes.forEach(function(c){
        var chip = document.createElement("div");
        chip.style.cssText = "display:flex;align-items:center;gap:5px;font-size:11px;color:#374151;";
        chip.innerHTML = '<span style="width:10px;height:10px;border-radius:50%;background:'+CORES[c]+';display:inline-block;"></span>'+c;
        legendaEl.appendChild(chip);
      });

      var xt = testPts[testSel];
      var r = knnPredict(xt);
      var vizIdx = r.viz.map(function(v){ return v.i; });

      // ---- SVG: pontos de treino, linhas para vizinhos, estrela de teste ----
      svg.innerHTML = "";
      // grade leve
      for(var g=1; g<4; g++){
        var lineV = svgNS("line");
        lineV.setAttribute("x1", g*25); lineV.setAttribute("y1", 0);
        lineV.setAttribute("x2", g*25); lineV.setAttribute("y2", 100);
        lineV.setAttribute("stroke", "#eee"); lineV.setAttribute("stroke-width", "0.4");
        svg.appendChild(lineV);
        var lineH = svgNS("line");
        lineH.setAttribute("x1", 0); lineH.setAttribute("y1", g*25);
        lineH.setAttribute("x2", 100); lineH.setAttribute("y2", g*25);
        lineH.setAttribute("stroke", "#eee"); lineH.setAttribute("stroke-width", "0.4");
        svg.appendChild(lineH);
      }
      // linhas ate os vizinhos (desenhadas antes dos pontos, para ficarem por baixo)
      vizIdx.forEach(function(i){
        var p = trainPts[i];
        var line = svgNS("line");
        line.setAttribute("x1", xt.x*100); line.setAttribute("y1", (1-xt.y)*100);
        line.setAttribute("x2", p.x*100); line.setAttribute("y2", (1-p.y)*100);
        line.setAttribute("stroke", CORES[p.cls]); line.setAttribute("stroke-width", "0.6");
        line.setAttribute("stroke-dasharray", "1.5,1"); line.setAttribute("opacity", "0.7");
        svg.appendChild(line);
      });
      // pontos de treino
      trainPts.forEach(function(p, i){
        var isViz = vizIdx.indexOf(i) !== -1;
        if(isViz){
          var halo = svgNS("circle");
          halo.setAttribute("cx", p.x*100); halo.setAttribute("cy", (1-p.y)*100);
          halo.setAttribute("r", 5); halo.setAttribute("fill", "none");
          halo.setAttribute("stroke", CORES[p.cls]); halo.setAttribute("stroke-width", "0.8");
          svg.appendChild(halo);
        }
        var c = svgNS("circle");
        c.setAttribute("cx", p.x*100); c.setAttribute("cy", (1-p.y)*100);
        c.setAttribute("r", 3.2);
        c.setAttribute("fill", CORES[p.cls]);
        c.setAttribute("stroke", "#fff"); c.setAttribute("stroke-width", "0.6");
        c.setAttribute("opacity", isViz ? "1" : "0.55");
        svg.appendChild(c);
      });
      // estrela de teste
      var correto = (r.pred === xt.cls);
      var estCor = correto ? "#16a34a" : "#dc2626";
      var halo2 = svgNS("circle");
      halo2.setAttribute("cx", xt.x*100); halo2.setAttribute("cy", (1-xt.y)*100);
      halo2.setAttribute("r", 5.5); halo2.setAttribute("fill", "#fff");
      halo2.setAttribute("stroke", estCor); halo2.setAttribute("stroke-width", "0.8");
      svg.appendChild(halo2);
      var txt = svgNS("text");
      txt.setAttribute("x", xt.x*100); txt.setAttribute("y", (1-xt.y)*100+1.8);
      txt.setAttribute("text-anchor", "middle"); txt.setAttribute("font-size", "6.5");
      txt.setAttribute("fill", estCor);
      txt.textContent = "★";
      svg.appendChild(txt);

      // ---- Distancias ordenadas ----
      distsEl.innerHTML = "";
r.ordenados.forEach(function(v, ord){
  var dentroK = ord < k;
  var row = document.createElement("div");
  row.style.cssText = "display:flex;justify-content:space-between;align-items:center;padding:2px 6px;border-radius:6px;" +
    (dentroK ? "background:"+CORES[v.p.cls]+"22;border:1px solid "+CORES[v.p.cls]+";" : "background:#f9fafb;border:1px solid #f1f1f1;color:#9ca3af;");
  row.innerHTML =
    '<span style="display:flex;align-items:center;gap:4px;">' +
      (dentroK ? '✓' : '\u00A0') +
      '<span title="position de lecture dans la liste d'entraînement d'origine — utilisée pour départager lorsque deux distances sont égales" ' +
        'style="background:#eee;color:#9ca3af;border-radius:3px;padding:0 3px;font-size:8.5px;cursor:help;">#' + (v.i+1) + '</span>' +
      ' ' + v.p.nome + ' <span style="color:'+CORES[v.p.cls]+';font-weight:700;">('+v.p.cls+')</span>' +
    '</span>' +
    '<span>d='+v.d.toFixed(4)+'</span>';
  distsEl.appendChild(row);
});

      // ---- Votacao ----
      votosEl.innerHTML = "";
      classes.forEach(function(c){
        var venceu = (c === r.pred);
        var empatou = r.empatados.length > 1 && r.empatados.indexOf(c) !== -1;
        var div = document.createElement("div");
        div.style.cssText = "text-align:center;border-radius:10px;padding:8px 14px;font-size:12px;" +
          (venceu ? "background:"+CORES[c]+"22;border:2px solid "+CORES[c]+";" : "background:#f9fafb;border:1px solid #e5e7eb;color:#9ca3af;");
        div.innerHTML = '<div style="font-weight:700;color:'+CORES[c]+';">'+c+'</div><div style="font-size:16px;font-weight:700;">'+r.votos[c]+'</div>' +
          (empatou ? '<div style="font-size:9px;color:#b91c1c;">empate</div>' : '');
        votosEl.appendChild(div);
      });
      var msgEmpate = r.empatados.length > 1 ? " (empate entre "+r.empatados.join(", ")+" — desempate pela ordem da lista de classes)" : "";
      previsaoEl.innerHTML = 'Classe prevista: <span style="color:'+CORES[r.pred]+';">'+r.pred+'</span>' + msgEmpate +
        ' &nbsp;|&nbsp; classe real: <span style="color:'+CORES[xt.cls]+';">'+xt.cls+'</span> ' + (correto ? '✅' : '❌');

      // ---- Matriz de confusao + acuracia sobre as 3 amostras de teste ----
      var cm = [[0,0,0],[0,0,0],[0,0,0]];
      var acertos = 0;
      var predsGlobais = [];
      testPts.forEach(function(tp){
        var rr = knnPredict(tp);
        predsGlobais.push(rr.pred);
        var iReal = classes.indexOf(tp.cls);
        var iPrev = classes.indexOf(rr.pred);
        cm[iReal][iPrev]++;
        if(rr.pred === tp.cls) acertos++;
      });
      var acc = acertos/testPts.length;

      var thead = '<tr><td></td>' + classes.map(function(c){ return '<td style="padding:4px 8px;color:'+CORES[c]+';font-weight:700;">'+c.slice(0,4)+'</td>'; }).join('') + '</tr>';
      var rows = classes.map(function(cReal, i){
        var cells = classes.map(function(cPrev, j){
          var v = cm[i][j];
          var diag = (i===j);
          var bg = v===0 ? '#fff' : (diag ? '#dcfce7' : '#fee2e2');
          return '<td style="padding:4px 10px;text-align:center;border:1px solid #e5e7eb;background:'+bg+';">'+v+'</td>';
        }).join('');
        return '<tr><td style="padding:4px 8px;color:'+CORES[cReal]+';font-weight:700;">'+cReal.slice(0,4)+'</td>'+cells+'</tr>';
      }).join('');
      cmEl.innerHTML = thead + rows;
      accEl.textContent = "Précision : " + acc.toFixed(4) + " (" + acertos + "/" + testPts.length + ")";

      dbg.textContent = "M="+metrica+" k="+k+" | teste_sel="+xt.nome+" | y_pred(todas)=["+predsGlobais.join(", ")+"]";
    }

    be.addEventListener("click", function(){ metrica="euclidiana"; render(); });
    bm.addEventListener("click", function(){ metrica="manhattan"; render(); });
    bk1.addEventListener("click", function(){ k=1; render(); });
    bk3.addEventListener("click", function(){ k=3; render(); });
    bk5.addEventListener("click", function(){ k=5; render(); });
    bt0.addEventListener("click", function(){ testSel=0; render(); });
    bt1.addEventListener("click", function(){ testSel=1; render(); });
    bt2.addEventListener("click", function(){ testSel=2; render(); });

    render();
  }
  function tryInit(){
    var root = document.getElementById("sim-ep0706");
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 7.6:** Simulateur EP07_06 : *Pipeline* k-NN Multi-Classe (vote, départage et matrice de confusion)


In [ ]:
%%writefile EP07_06.py
# Code Python

In [ ]:
TestSuite("EP07_06.py").run()

### EP07_07 ⚫ Classification réelle d’une mosaïque de textures via LBP + k-NN

Dans les exercices précédents, le descripteur LBP (**EP07_04**) et le classifieur k-NN multiclasse (**EP07_06**) ont été étudiés séparément, toujours à partir de données déjà fournies en entrée — voisinages $3\times3$ isolés ou histogrammes préalablement extraits. Dans cet exercice de clôture du chapitre, le programme devra **lire une image réelle**, au format **PGM ASCII (P2)**, calculer le descripteur LBP directement à partir des pixels, puis classer chaque région au moyen du k-NN, reproduisant ainsi, à échelle réduite, le flux complet d’un système de reconnaissance de textures. Cette approche anticipe également l’idée de **classification par mosaïque de régions**, liée à la segmentation sémantique étudiée dans un chapitre ultérieur.

Le simulateur interactif de l’**EP07_06** n’utilisait que trois classes (`granular`, `listrada` et `manchada`) représentées par des points bidimensionnels fictifs. Dans cet exercice, une quatrième classe, **xadrez**, est ajoutée, et les points sont remplacés par des histogrammes LBP extraits d’une image réelle.

L’image d’entrée est une **mosaïque** formée d’une grille $G\times G$ de blocs carrés de $S\times S$ pixels. Chaque bloc contient un échantillon de l’une des quatre classes de texture synthétique du chapitre : **granular**, **listrada**, **manchada** ou **xadrez** (motif en damier avec intensités alternées). Comme dans les autres exercices du livre, le chargement de l’image est réalisé par la fonction didactique `mm.readImg`.

> ### 💡 Pourquoi une mosaïque unique, et non plusieurs images ?
>
> L’entrée regroupe les $G \times G$ échantillons de texture dans un seul fichier **PGM**, uniquement pour simplifier la lecture des données et éviter l’ouverture de plusieurs fichiers. Pour l’algorithme, cela ne modifie pas le traitement : chaque bloc est traité de manière indépendante, comme s’il s’agissait d’une image isolée.
> La seule exception est l’**exclusion de la bordure** (point 4 ci-dessous).

#### 📋 Directives d’implémentation

1. **Lecture des dimensions de l’image**

   Lire, via l’entrée standard, deux lignes contenant respectivement le nombre de lignes $L$ et le nombre de colonnes $C$ de la mosaïque (tous deux multiples de la taille de bloc $S$, avec $L=C$).

2. **Chargement de l’image**

   Utiliser la fonction didactique

   ```python
   f = mm.readImg(L, C)
   ```

   pour lire les $L \times C$ valeurs d’intensité (niveaux de gris, `uint8`) de la mosaïque.

3. **Paramètres de la grille**

   Lire l’entier $G$ (nombre de blocs par côté) et l’entier $S$ (taille du côté de chaque bloc, en pixels), satisfaisant $L = C = G \times S$.

4. **Calcul du code LBP par pixel**

   Pour chaque pixel **intérieur** de l’image (c’est-à-dire ne se trouvant pas sur la bordure globale de `f` — ligne ou colonne $0$ ou $L-1$/$C-1$), calculer le code LBP avec $P=8$ voisins et rayon $R=1$, en parcourant les voisins dans le sens **horaire** à partir du coin supérieur gauche, exactement comme dans l’EP07_04 : `[lin-1][col-1]`, `[lin-1][col]`, `[lin-1][col+1]`, `[lin][col+1]`, `[lin+1][col+1]`, `[lin+1][col]`, `[lin+1][col-1]`, `[lin][col-1]`.

   Les pixels sur la bordure globale de l’image **n’ont pas** de voisinage complet et doivent être **ignorés** (ils ne contribuent à aucun histogramme). Cela inclut les pixels de bordure qui tombent à l’intérieur d’un bloc (l’exclusion est toujours relative à la bordure de l’image entière, et non à celle de chaque bloc).

5. **Histogramme LBP uniforme par bloc (10 compartiments)**

   Pour chaque bloc $(i,j)$ de la grille ($i,j = 0,\ldots,G-1$), accumuler, parmi ses pixels valides (point 4), un histogramme $H^{(i,j)}$ de $10$ compartiments :

   * En considérant la séquence circulaire de bits $s_0,\ldots,s_7$ du pixel (même règle de transitions que l’EP07_04) : si le nombre de transitions est $\le 2$ (motif **uniforme**), le pixel contribue au compartiment $\operatorname{popcount}(s_0,\ldots,s_7) \in \{0,\ldots,8\}$ (nombre de bits égaux à `1`) ;
   * Sinon (motif **non uniforme**), le pixel contribue au compartiment $9$.

   À la fin, normaliser l’histogramme de chaque bloc en le divisant par le nombre de pixels valides qu’il contient, obtenant $\hat H^{(i,j)}$, avec $\sum_{b=0}^{9} \hat H^{(i,j)}[b] = 1$.

6. **Prototypes d’apprentissage**

   Lire l’entier $Ncl$ (nombre de classes) suivi de $Ncl$ noms de classes (ordre définissant la matrice de confusion et le départage des votes, comme dans l’EP07_06) ; ensuite, lire la *chaîne* $M$ (métrique : `euclidiana` ou `manhattan`) et l’entier impair $k$ ; enfin, lire l’entier $N$ (nombre de prototypes) et, pour chacun, le nom de la classe suivi de $10$ valeurs réelles (histogramme prototype déjà normalisé).

7. **Classification k-NN de chaque bloc**

   Pour chaque bloc, calculer la distance de $\hat H^{(i,j)}$ à chacun des $N$ prototypes, en utilisant la métrique $M$ (mêmes formules que l’EP07_06). Sélectionner les $k$ prototypes les plus proches (départage des distances par l’ordre de lecture des prototypes) et classer selon la classe majoritaire (départage des votes par l’ordre des classes du point 6).

8. **Étiquettes réelles et évaluation**

   Lire, sur une seule ligne, les $G \times G$ noms de classes **réels** de chaque bloc, dans l’ordre de lecture par ligne de la grille (bloc $(0,0)$, $(0,1)$, …, $(0,G-1)$, $(1,0)$, …). Construire la matrice de confusion $Ncl \times Ncl$ (ligne = classe réelle, colonne = classe prédite) et l’exactitude globale.

9. **Sortie**

   Imprimer, pour chaque bloc (dans le même ordre de lecture que les étiquettes réelles du point 8), la classe prédite. Ensuite, imprimer la matrice de confusion (une ligne par classe réelle, dans l’ordre du point 6). Enfin, imprimer l’exactitude, arrondie à 4 décimales.

#### 📌 Contraintes informatiques

* **Descripteur fixe :** $P=8$, $R=1$ et $10$ compartiments (selon le point 5) sont fixes dans cet exercice — ils ne sont pas lus depuis l’entrée.
* **Exclusion de bordure globale, non de bloc :** un pixel à la limite entre deux blocs, mais à l’intérieur de l’image, est valide et contribue normalement à l’histogramme du bloc auquel il appartient.
* **Ordre de lecture comme critère de départage :** tant le départage des distances (point 7) que celui des votes (point 7) suivent exactement les mêmes conventions que l’EP07_01 et l’EP07_06.
* **Prototypes en entrée, non appris :** contrairement au Projet Pratique 2, les histogrammes d’apprentissage sont fournis directement en entrée ; le programme ne doit pas générer de textures synthétiques.

#### 🧠 Fondements théoriques

| Étape de l’exercice | Étape correspondante dans le chapitre |
|---|---|
| Lecture de l’image via `mm.readImg` | Acquisition de l’image dans le *pipeline* de reconnaissance de formes |
| Code LBP par pixel (EP07_04) | `local_binary_pattern(image, P=8, R=1, method="uniform")` |
| Histogramme de 10 compartiments par bloc | Fonction `descritor_lbp` du Projet Pratique 2 (`bins=10`, `range=(0, P+2)`) |
| Classification k-NN avec métrique sélectionnable (EP07_06) | `KNeighborsClassifier` entraîné sur `X_textura` |
| Matrice de confusion $Ncl\times Ncl$ et exactitude | `confusion_matrix` et `accuracy_score` sur `yt_teste` |

Cet exercice met en évidence, avec des pixels réels plutôt que des valeurs synthétiques, une limitation discutée dans la section finale du chapitre : des classes de texture visuellement distinctes pour un observateur humain — comme **granular** et **manchada** — peuvent produire des histogrammes LBP similaires lorsque le voisinage considéré est petit ($R=1$), car toutes deux présentent une fréquence élevée de motifs non uniformes à l’échelle d’un seul pixel. La classe **xadrez**, quant à elle, possédant des bordures régulières et répétitives, tend à être séparée plus facilement. On s’attend à ce que la matrice de confusion produite reflète exactement ce schéma de confusion partielle.

#### 📦 Spécification d’entrée et de sortie (VPL)

**Entrée :**

```
L
C
[matrice L x C de l’image]
G S
Ncl nom_classe_1 ... nom_classe_Ncl
M k
N
nom_classe h0 h1 ... h9      (répétée N fois)
etiquette(0,0) etiquette(0,1) ... etiquette(G-1,G-1)
```

**Sortie :**

* $G \times G$ lignes avec la classe prédite de chaque bloc, dans l’ordre de lecture de la grille.
* $Ncl$ lignes avec la matrice de confusion (une ligne par classe réelle, valeurs séparées par des espaces).
* Dernière ligne : `Acuracia: <valeur>`.

#### 📌 Exemple (vérification manuelle)

Pour vérifier l’implémentation du descripteur avant de la tester sur une mosaïque complète, considérons une image $6\times6$ **homogène**, avec tous les pixels d’intensité $100$, traitée comme un seul bloc ($G=1$, $S=6$). Comme tout pixel intérieur a ses 8 voisins d’intensité égale à celle du centre ($g_p \ge g_c$ dans tous les cas), tous les bits $s_p$ valent `1`, le nombre de transitions est $0$ (uniforme) et le compartiment est $\operatorname{popcount}(11111111)=8$. L’histogramme de l’unique bloc est donc `0 0 0 0 0 0 0 0 1 0`.

| Entrée (résumée) | Sortie | Observation |
|---|---|---|
| 6<br>6<br>[36 valeurs égales à 100]<br>1 6<br>2 uniforme outra<br>euclidiana 1<br>2<br>uniforme 0 0 0 0 0 0 0 0 1 0<br>outra 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1<br>uniforme | uniforme<br>1 0<br>0 0<br>Acuracia: 1.0000 | La distance du bloc au prototype `uniforme` est exactement $0$ ; la classe `outra` n’apparaît pas dans l’étiquette réelle, c’est pourquoi sa ligne dans la matrice de confusion est nulle. |

#### 📌 Fichiers de référence (.pgm)

Pour le débogage local, deux mosaïques de test au format ASCII P2 sont fournies (annexées à cette livraison ; lors de leur intégration au référentiel du chapitre, enregistrez-les dans `all/cap07/dados/EP07/`) :

* 📥 **Cas 1 — Mosaïque simple (`Caso1_Mosaico_Simples.pgm`)** : grille $2\times2$ de blocs de $24\times24$ pixels, un échantillon de chacune des quatre classes, avec faible bruit — utile pour valider la lecture de l’image et la logique de classification dans un scénario contrôlé.
* 📥 **Cas 2 — Mosaïque mixte (`Caso2_Mosaico_Misto.pgm`)** : grille $3\times3$ de blocs de $16\times16$ pixels, avec classes répétées et plus grande variabilité — scénario dans lequel la confusion entre **granular** et **manchada** discutée dans les Fondements théoriques tend à se manifester.

La [Figure 7.7](#fig-07-ep07) affiche les deux mosaïques, pour une inspection visuelle avant l’implémentation.

In [ ]:
import os
import urllib.request
import numpy as np

def garantir_e_baixar_arquivo(nome_arquivo):
    diretorio_local = "dados/EP07"
    caminho_local = os.path.join(diretorio_local, nome_arquivo)
    
    # Créer le répertoire local s'il n'existe pas
    if not os.path.exists(diretorio_local):
        os.makedirs(diretorio_local)
        
    # Si le fichier n'existe pas localement, le télécharger depuis le dépôt distant
    if not os.path.exists(caminho_local):
        url_base = "https://raw.githubusercontent.com/fzampirolli/"
        url_base += "pdi-vc/master/all/cap07/dados/EP07"
        url_arquivo = f"{url_base}/{nome_arquivo}"
        print(f"Téléchargement {nome_arquivo} depuis GitHub...")
        try:
            urllib.request.urlretrieve(url_arquivo, caminho_local)
        except Exception as e:
            raise IOError(f"Erro ao baixar {nome_arquivo} do GitHub. ",
                          "Verifique a conexão ou a URL. Detalhes: {e}")
            
    return caminho_local

def ler_pgm_p2(caminho):
    with open(caminho) as f:
        linhas = [l for l in f.read().split() if l]
    assert linhas[0] == "P2"
    C, L = int(linhas[1]), int(linhas[2])
    maxv = int(linhas[3])
    valores = list(map(int, linhas[4:4 + L * C]))
    return np.array(valores, dtype=np.uint8).reshape(L, C)

# Garantit le téléchargement et obtient le chemin correct
arq_caso1 = garantir_e_baixar_arquivo("Caso1_Mosaico_Simples.pgm")
arq_caso2 = garantir_e_baixar_arquivo("Caso2_Mosaico_Misto.pgm")

# Lit les matrices PGM
caso1 = ler_pgm_p2(arq_caso1)
caso2 = ler_pgm_p2(arq_caso2)

mm.show(
    [caso1, caso2],
    titles=[
        "Cas 1 : Mosaïque Simple\n(blocs 2x2, 1 échantillon/classe)",
        "Cas 2 : Mosaïque Mixte\n(blocs 3x3, classes répétées)",
    ],
    cols=2,
    figsize=(8, 4),
)

**Figure 7.7:** Mosaïques de référence (format PGM ASCII) utilisées dans l


In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0707" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">  
<style>
  #sim-ep0707 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0707 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #ede6d8; background: #f3efe6; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0707 button:hover { background: #e8e0cf; }
  #sim-ep0707 button.sim-ep0707_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0707_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #ede6d8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0707_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulateur EP07_07 : Classification de mosaïque via LBP + k-NN</span>
  <span class="sim-ep0707_pill">⚫ pipeline complet</span>
</div>


  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
Mosaïque 3x3 de blocs 12x12 (L=C=36). LBP (P=8,R=1) calculé pixel par pixel, avec exclusion de la bordure globale.      Ajustez k et la métrique et observez la classification de chaque bloc face à 8 prototypes (2 par classe).
   
   </p>
     
<div style="background:#fff3cd;border:1px solid #ffe69c;border-radius:8px;padding:8px 12px;margin-bottom:12px;font-size:11px;color:#7a5c00;">
  ⚠️ Textures synthétiques générées par code, pas les fichiers .pgm réels de l'EP07_07. Utilisez ce simulateur pour comprendre le flux de l'algorithme, pas comme référence de difficulté entre les classes.
</div>
     
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;display:flex;gap:24px;flex-wrap:wrap;align-items:center;">
      <div style="flex:1;min-width:180px;">
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
          <label style="font-size:12px;font-weight:bold;color:#2980b9;">k (nombre de voisins)</label>
          <span id="ep0707_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">1</span>
        </div>
<input id="ep0707_sl" style="width:100%;accent-color:#2980b9;" max="5" min="1" step="2" type="range" value="1">
      </div>
      <div>
        <label style="font-size:12px;font-weight:bold;color:#2980b9;display:block;margin-bottom:6px;">Métrique</label>
        <select id="ep0707_metric" style="font-size:12px;padding:4px 8px;border-radius:6px;border:1px solid #ccc;">
          <option value="euclidiana">euclidienne</option>
          <option value="manhattan">manhattan</option>
        </select>
      </div>
    </div>

    <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:flex-start;">
      <canvas id="ep0707_canvas" style="border-radius:8px;border:1px solid #ccc;"></canvas>
      <div id="ep0707_grid" style="flex:1;min-width:220px;display:grid;grid-template-columns:repeat(3,1fr);gap:8px;"></div>
    </div>

    <div id="ep0707_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;white-space:pre-line;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var S = 12, G = 3, L = G * S, SCALE = 5;   // bloco maior reduz o vazamento de borda; SCALE ajustado p/ manter o canvas ~180px
    var classesOrder = ["granular", "listrada", "manchada", "xadrez"];
    // Grade 3x3 com classes repetidas, análoga ao Caso 2 do enunciado
    var layout = [
      "granular", "listrada", "manchada",
      "xadrez",   "granular", "manchada",
      "listrada", "xadrez",   "granular"
    ];

    // --- Geração determinística de textura por pixel (didática, não os PGMs reais) ---
    function h(a, b, phase){
      var v = Math.sin((a + phase) * 12.9898 + (b + phase * 0.7) * 78.233 + phase * 3.1) * 43758.5453;
      return v - Math.floor(v);
    }
    function texturePixel(cls, r, c, phase){
      phase = phase || 0;
      switch(cls){
        case "granular": return h(r, c, phase) < 0.5 ? 220 : 30;
        case "listrada": return ((c + Math.floor(phase * 2)) % 4) < 2 ? 220 : 30;
        case "manchada": return h(Math.floor(r / 3), Math.floor(c / 3), phase) < 0.5 ? 200 : 60;
        case "xadrez":   return ((Math.floor(r / 2) + Math.floor(c / 2)) % 2 === 0) ? 230 : 20;
      }
    }

    function buildImage(){
      var img = [];
      for(var r = 0; r < L; r++){
        var row = [];
        for(var c = 0; c < L; c++){
          var bi = Math.floor(r / S), bj = Math.floor(c / S);
          row.push(texturePixel(layout[bi * G + bj], r, c, 0));
        }
        img.push(row);
      }
      return img;
    }

    // --- LBP: P=8, R=1, sentido horário, s_p = 1 se vizinho >= centro ---
    function lbpBin(patch, r, c){
      var center = patch[r][c];
      var neigh = [
        patch[r-1][c-1], patch[r-1][c], patch[r-1][c+1],
        patch[r][c+1],
        patch[r+1][c+1], patch[r+1][c], patch[r+1][c-1],
        patch[r][c-1]
      ];
      var bits = neigh.map(function(v){ return v >= center ? 1 : 0; });
      var trans = 0;
      for(var i = 0; i < 8; i++){ if(bits[i] !== bits[(i+1) % 8]) trans++; }
      if(trans <= 2) return bits.reduce(function(a,b){ return a+b; }, 0); // popcount 0..8
      return 9; // não uniforme
    }

    // Histograma de um patch isolado (usado para gerar protótipos), excluindo apenas a borda do patch
    function computeLBPHist(patch){
      var n = patch.length, m = patch[0].length;
      var hist = new Array(10).fill(0), count = 0;
      for(var r = 1; r < n - 1; r++){
        for(var c = 1; c < m - 1; c++){
          hist[lbpBin(patch, r, c)]++;
          count++;
        }
      }
      for(var k = 0; k < 10; k++) hist[k] = count > 0 ? hist[k] / count : 0;
      return hist;
    }

    // Histogramas por bloco da imagem completa, excluindo só a borda global (item 4/5 do enunciado)
    function computeMosaicHistograms(img){
      var hists = [], counts = [];
      for(var i = 0; i < G*G; i++){ hists.push(new Array(10).fill(0)); counts.push(0); }
      for(var r = 1; r < L - 1; r++){
        for(var c = 1; c < L - 1; c++){
          var bin = lbpBin(img, r, c);
          var idx = Math.floor(r/S) * G + Math.floor(c/S);
          hists[idx][bin]++;
          counts[idx]++;
        }
      }
      for(var b = 0; b < hists.length; b++){
        for(var k = 0; k < 10; k++) hists[b][k] = counts[b] > 0 ? hists[b][k] / counts[b] : 0;
      }
      return hists;
    }

    // --- Protótipos: 2 por classe (N=8), ordem de leitura fixa (usada no desempate) ---
    var prototypes = [];
    classesOrder.forEach(function(cls){
      [0, 5].forEach(function(phase){
        var Sp = S + 2, patch = [];
        for(var r = 0; r < Sp; r++){
          var row = [];
          for(var c = 0; c < Sp; c++) row.push(texturePixel(cls, r, c, phase));
          patch.push(row);
        }
        prototypes.push({ classe: cls, hist: computeLBPHist(patch) });
      });
    });

    function dist(u, v, metric){
      var s = 0;
      for(var i = 0; i < u.length; i++){
        s += metric === "euclidiana" ? (u[i]-v[i])*(u[i]-v[i]) : Math.abs(u[i]-v[i]);
      }
      return metric === "euclidiana" ? Math.sqrt(s) : s;
    }

    // Desempate de distância: ordem de leitura dos protótipos. Desempate de votação: ordem das classes.
    function classify(hist, k, metric){
      var cand = prototypes.map(function(p, idx){ return { classe: p.classe, d: dist(hist, p.hist, metric), idx: idx }; });
      cand.sort(function(a, b){ return a.d !== b.d ? a.d - b.d : a.idx - b.idx; });
      var viz = cand.slice(0, k);
      var votos = {};
      viz.forEach(function(v){ votos[v.classe] = (votos[v.classe] || 0) + 1; });
      var melhor = null, melhorN = -1;
      classesOrder.forEach(function(c){
        var n = votos[c] || 0;
        if(n > melhorN){ melhorN = n; melhor = c; }
      });
      return melhor;
    }

    var canvas = root.querySelector('#ep0707_canvas');
    canvas.width = L * SCALE; canvas.height = L * SCALE;
    var ctx = canvas.getContext('2d');
    var slK = root.querySelector('#ep0707_sl');
    var vlK = root.querySelector('#ep0707_vl');
    var selMetric = root.querySelector('#ep0707_metric');
    var gridEl = root.querySelector('#ep0707_grid');
    var dbg = root.querySelector('#ep0707_debug');

    var img = buildImage();
    var hists = computeMosaicHistograms(img);

    function render(){
      var k = parseInt(slK.value);
      var metric = selMetric.value;
      vlK.textContent = k;

      var preds = [];
      for(var idx = 0; idx < G*G; idx++) preds.push(classify(hists[idx], k, metric));

      var confusion = classesOrder.map(function(){ return new Array(classesOrder.length).fill(0); });
      var acertos = 0;
      for(var i2 = 0; i2 < G*G; i2++){
        var ri = classesOrder.indexOf(layout[i2]);
        var pi = classesOrder.indexOf(preds[i2]);
        confusion[ri][pi]++;
        if(layout[i2] === preds[i2]) acertos++;
      }
      var acc = acertos / (G*G);

      // Desenha a imagem real em tons de cinza
      for(var r = 0; r < L; r++){
        for(var c = 0; c < L; c++){
          var v = img[r][c];
          ctx.fillStyle = 'rgb(' + v + ',' + v + ',' + v + ')';
          ctx.fillRect(c*SCALE, r*SCALE, SCALE, SCALE);
        }
      }
      // Contorna cada bloco: verde = acerto, vermelho = erro
      for(var idx3 = 0; idx3 < G*G; idx3++){
        var bi = Math.floor(idx3 / G), bj = idx3 % G;
        ctx.strokeStyle = (preds[idx3] === layout[idx3]) ? '#10b981' : '#f43f5e';
        ctx.lineWidth = 2;
        ctx.strokeRect(bj*S*SCALE + 1, bi*S*SCALE + 1, S*SCALE - 2, S*SCALE - 2);
      }

      // Grade textual de apoio
      gridEl.innerHTML = '';
      for(var idx4 = 0; idx4 < G*G; idx4++){
        var ok = preds[idx4] === layout[idx4];
        var card = document.createElement('div');
        card.style.cssText = 'border-radius:8px;padding:6px;text-align:center;font-size:10px;border:2px solid ' + (ok ? '#10b981' : '#f43f5e') + ';';
        card.innerHTML = 'Real: ' + layout[idx4] + '<br><b style="color:' + (ok ? '#059669' : '#e11d48') + '">Pred: ' + preds[idx4] + (ok ? ' ✅' : ' ❌') + '</b>';
        gridEl.appendChild(card);
      }

      // Saída no mesmo formato do programa (itens 7-9 do enunciado)
      var linhas = [];
      linhas.push('Classes preditas (ordem de leitura da grade):');
      linhas.push(preds.join(' '));
      linhas.push('');
      linhas.push('Matriz de confusão (linhas=real, colunas=predita; ordem ' + classesOrder.join(',') + '):');
      confusion.forEach(function(lin){ linhas.push(lin.join(' ')); });
      linhas.push('');
      linhas.push('Acuracia: ' + acc.toFixed(4));
      dbg.textContent = linhas.join('\\n');
    }

    slK.addEventListener('input', render);
    selMetric.addEventListener('change', render);
    render();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0707');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figure 7.8:** Simulateur EP07_07 : Classification d


In [ ]:
%%writefile EP07_07.py
# Code Python

In [ ]:
TestSuite("EP07_07.py").run()